# Notebook 16 — Final Out-of-Time Test (Version 2 lagged-bank main analysis)

This notebook performs the **one-time Version 2 final evaluation** after Notebook 15 has locked all six model specifications, predictor sets, preprocessing rules, hyperparameters and pooled-validation F1 thresholds.

The fixed test calendar is 2020Q1–2025Q2. This calendar was examined previously under the archived unlagged Version 1, so it is not described as never viewed. The defensible claim is narrower: the exact-t−1 lagged-bank design and the complete Version 2 selection procedure were fixed before this Version 2 evaluation. No test result may tune, select, relock or recalibrate a model.

Bank predictors are measured at `source_report_date = t−1`; the outcome remains failure in `(t, t+365]`. The realised predictor-date-to-failure horizon is therefore reported explicitly and is longer than in Version 1.

In [ ]:
# Resolve the immutable Version 2 handoff and lock the one-time evaluation contract

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib.util
import json
import math
import os
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn
import xgboost as xgb
from IPython.display import display
from scipy.stats import ks_2samp
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score, brier_score_loss, confusion_matrix, f1_score,
    log_loss, precision_score, recall_score, roc_auc_score,
)
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier


COLAB_MYDRIVE = Path("/content/drive/MyDrive")
try:
    from google.colab import drive
except ImportError:
    drive = None

if COLAB_MYDRIVE.is_dir():
    print("Google Drive is already mounted at /content/drive.")
elif drive is not None:
    try:
        drive.mount("/content/drive", force_remount=False)
    except (ValueError, OSError) as exc:
        if not COLAB_MYDRIVE.is_dir():
            raise RuntimeError("Google Drive could not be mounted. Authorise it and rerun Cell 1.") from exc


DEFAULT_PROJECT_ROOT = Path(
    os.environ.get("DISSERTATION_PROJECT_ROOT", "/content/drive/MyDrive/Data Final")
).expanduser()
explicit_config = os.environ.get("V2_CONFIG_PATH")
CONFIG_PATH = (
    Path(explicit_config).expanduser()
    if explicit_config
    else DEFAULT_PROJECT_ROOT / "Version 2 Main Analysis" / "config.py"
)
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f"Shared Version 2 config not found: {CONFIG_PATH}")

config_spec = importlib.util.spec_from_file_location("v2_config", CONFIG_PATH)
config = importlib.util.module_from_spec(config_spec)
config_spec.loader.exec_module(config)
REQUIRED_CONFIG_FIELDS = {
    "ANALYSIS_VERSION", "VERSION_2_MAIN_TIMING_POLICY", "V2_OUTPUT_ROOT",
    "FINAL_TEST_DESIGN", "TARGET_HORIZON_DAYS", "RANDOM_SEED",
    "MISSING_DATA_POLICY", "HISTORY_POLICY",
}
missing_config_fields = sorted(REQUIRED_CONFIG_FIELDS.difference(dir(config)))
if missing_config_fields:
    raise AttributeError("Shared config missing: " + ", ".join(missing_config_fields))

ANALYSIS_VERSION = str(config.ANALYSIS_VERSION)
TIMING_POLICY = dict(config.VERSION_2_MAIN_TIMING_POLICY)
FINAL_TEST_DESIGN = dict(config.FINAL_TEST_DESIGN)
TARGET_HORIZON_DAYS = int(config.TARGET_HORIZON_DAYS)
RANDOM_SEED = int(config.RANDOM_SEED)
MISSING_DATA_POLICY = dict(config.MISSING_DATA_POLICY)
HISTORY_POLICY = dict(config.HISTORY_POLICY)
V2_OUTPUT_ROOT = Path(config.V2_OUTPUT_ROOT).expanduser()
PROJECT_ROOT = Path(getattr(config, "PROJECT_ROOT", CONFIG_PATH.parent.parent)).expanduser()
V2_ROOT = Path(getattr(config, "V2_ROOT", CONFIG_PATH.parent)).expanduser()

TARGET = "failed_1yr"
PANEL_KEYS = ["report_date", "cert", "idrssd"]
HISTORY_INDICATOR = str(HISTORY_POLICY["indicator_model_name"])
TOP_RISK_PERCENTAGES = (0.01, 0.05, 0.10, 0.20)
WINSOR_LOWER_QUANTILE = 0.01
WINSOR_UPPER_QUANTILE = 0.99
BOOTSTRAP_SEED = int(getattr(config, "BOOTSTRAP_SEED", 20260816))
BOOTSTRAP_REPLICATES = int(getattr(config, "BOOTSTRAP_REPLICATES", 1000))
BOOTSTRAP_CONFIDENCE_LEVEL = 0.95
EXPECTED_TEST_ROWS = 92229
EXPECTED_TEST_POSITIVES = 40
EXPECTED_TEST_FAILED_BANKS = 13
EXPECTED_TEST_QUARTERS = 22
EXPECTED_MODEL_IDS = {
    "logistic_regression__bank_only", "logistic_regression__bank_plus_macro",
    "random_forest__bank_only", "random_forest__bank_plus_macro",
    "xgboost__bank_only", "xgboost__bank_plus_macro",
}


def quarter_end(label):
    return pd.Period(str(label), freq="Q").end_time.normalize()


def deduplicate_paths(paths):
    result = []
    for item in paths:
        item = Path(item).expanduser()
        if item not in result:
            result.append(item)
    return result


def bundle_complete(root, relative_paths):
    return all((root / relative).is_file() for relative in relative_paths)


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def passed_values(series):
    return series.astype(str).str.strip().str.lower().isin({"true", "1", "pass"})


def is_relative_to(path, root):
    try:
        Path(path).resolve().relative_to(Path(root).resolve())
        return True
    except ValueError:
        return False


def to_builtin(value):
    if isinstance(value, dict):
        return {str(key): to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(item) for item in value]
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, np.generic):
        return value.item()
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    return value


def strict_json_dump(payload, path):
    temp_path = path.with_suffix(path.suffix + ".tmp")
    with open(temp_path, "w", encoding="utf-8") as handle:
        json.dump(to_builtin(payload), handle, indent=2, sort_keys=True, allow_nan=False)
    temp_path.replace(path)


data_bundle_paths = [
    Path("09_Final_Modelling_Panel") / "v2_final_modelling_panel_2001Q2_2025Q2.parquet",
    Path("09_Final_Modelling_Panel") / "v2_positive_predictor_availability_margin_audit.parquet",
    Path("09_Final_Modelling_Panel") / "v2_cert_mismatch_allowlist.csv",
    Path("09_Final_Modelling_Panel") / "v2_cert_mismatch_exclusion_list.csv",
    Path("09_Final_Modelling_Panel") / "cell_08_final_validation_gate.csv",
    Path("10_Chronological_Validation_Design") / "v2_chronological_row_membership.parquet",
    Path("10_Chronological_Validation_Design") / "v2_chronological_period_summary.csv",
    Path("10_Chronological_Validation_Design") / "cell_07_final_validation_gate.csv",
]
lock_bundle_paths = [
    Path("15_Validation_Comparison_and_Model_Lock") / "v2_model_lock_manifest.json",
    Path("15_Validation_Comparison_and_Model_Lock") / "v2_six_specification_mean_validation_comparison.csv",
    Path("15_Validation_Comparison_and_Model_Lock") / "cell_02_selected_prediction_source_manifest.csv",
    Path("15_Validation_Comparison_and_Model_Lock") / "cell_07_final_validation_gate.csv",
]
candidate_roots = deduplicate_paths([
    V2_OUTPUT_ROOT / ANALYSIS_VERSION,
    V2_OUTPUT_ROOT,
    V2_ROOT / "Notebook Outputs" / ANALYSIS_VERSION,
    V2_ROOT / "Notebook Outputs",
    PROJECT_ROOT / "Notebook Outputs" / ANALYSIS_VERSION,
    PROJECT_ROOT / "Notebook Outputs",
    DEFAULT_PROJECT_ROOT / "Notebook Outputs" / ANALYSIS_VERSION,
    DEFAULT_PROJECT_ROOT / "Notebook Outputs",
])
data_roots = [root for root in candidate_roots if bundle_complete(root, data_bundle_paths)]
lock_roots = [root for root in candidate_roots if bundle_complete(root, lock_bundle_paths)]
if not data_roots:
    raise FileNotFoundError("No complete Version 2 Notebook 09–10 handoff bundle was found.")
if not lock_roots:
    raise FileNotFoundError("No complete Version 2 Notebook 15 model-lock bundle was found.")

DATA_OUTPUT_ROOT = data_roots[0]
LOCK_OUTPUT_ROOT = lock_roots[0]
NB09_OUTPUT_DIR = DATA_OUTPUT_ROOT / "09_Final_Modelling_Panel"
NB10_OUTPUT_DIR = DATA_OUTPUT_ROOT / "10_Chronological_Validation_Design"
NB15_OUTPUT_DIR = LOCK_OUTPUT_ROOT / "15_Validation_Comparison_and_Model_Lock"
NB16_OUTPUT_DIR = LOCK_OUTPUT_ROOT / "16_Final_Out_of_Time_Test"
MODEL_DIR = NB16_OUTPUT_DIR / "final_models"
PREDICTION_DIR = NB16_OUTPUT_DIR / "final_test_predictions"
PREPROCESSING_DIR = NB16_OUTPUT_DIR / "final_training_preprocessing"
CHECKPOINT_DIR = NB16_OUTPUT_DIR / "fit_checkpoints"
FIGURE_DIR = NB16_OUTPUT_DIR / "figures"
for directory in [NB16_OUTPUT_DIR, MODEL_DIR, PREDICTION_DIR, PREPROCESSING_DIR, CHECKPOINT_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

FINAL_PANEL_PATH = NB09_OUTPUT_DIR / "v2_final_modelling_panel_2001Q2_2025Q2.parquet"
HORIZON_AUDIT_PATH = NB09_OUTPUT_DIR / "v2_positive_predictor_availability_margin_audit.parquet"
CERT_ALLOWLIST_PATH = NB09_OUTPUT_DIR / "v2_cert_mismatch_allowlist.csv"
CERT_EXCLUSION_PATH = NB09_OUTPUT_DIR / "v2_cert_mismatch_exclusion_list.csv"
NB09_FINAL_GATE_PATH = NB09_OUTPUT_DIR / "cell_08_final_validation_gate.csv"
MEMBERSHIP_PATH = NB10_OUTPUT_DIR / "v2_chronological_row_membership.parquet"
PERIOD_SUMMARY_PATH = NB10_OUTPUT_DIR / "v2_chronological_period_summary.csv"
NB10_FINAL_GATE_PATH = NB10_OUTPUT_DIR / "cell_07_final_validation_gate.csv"
MODEL_LOCK_PATH = NB15_OUTPUT_DIR / "v2_model_lock_manifest.json"
VALIDATION_COMPARISON_PATH = NB15_OUTPUT_DIR / "v2_six_specification_mean_validation_comparison.csv"
VALIDATION_SOURCE_MANIFEST_PATH = NB15_OUTPUT_DIR / "cell_02_selected_prediction_source_manifest.csv"
NB15_FINAL_GATE_PATH = NB15_OUTPUT_DIR / "cell_07_final_validation_gate.csv"
RUN_MARKER_PATH = NB16_OUTPUT_DIR / "FINAL_TEST_RUN_MARKER.json"

required_paths = [
    CONFIG_PATH, FINAL_PANEL_PATH, HORIZON_AUDIT_PATH, CERT_ALLOWLIST_PATH,
    CERT_EXCLUSION_PATH, NB09_FINAL_GATE_PATH, MEMBERSHIP_PATH, PERIOD_SUMMARY_PATH,
    NB10_FINAL_GATE_PATH, MODEL_LOCK_PATH, VALIDATION_COMPARISON_PATH,
    VALIDATION_SOURCE_MANIFEST_PATH, NB15_FINAL_GATE_PATH,
]
input_hashes_before = {str(path): sha256_file(path) for path in required_paths}

evaluation_contract = pd.DataFrame({
    "item": [
        "analysis", "training sample", "purge", "test sample", "test-calendar disclosure",
        "models", "preprocessing", "class weighting", "selection", "threshold",
        "capacity", "bootstrap", "conversion identity", "baseline", "calibration",
        "effective horizon", "post-test changes",
    ],
    "locked_rule": [
        "Version 2 exact-t-1 lagged-bank main analysis",
        "eligible reference rows 2001Q2–2018Q4 only",
        "exclude all 2019 reference rows from fitting and evaluation",
        "fixed 2020Q1–2025Q2 calendar, evaluated once within Version 2",
        "the same calendar was examined under archived unlagged Version 1; V2 design and selection were fixed before V2 evaluation",
        "all six Notebook 15 locked specifications; no tuning",
        "refit medians, 1st/99th percentile bounds and LR scaler on final training only",
        "final-training negative/positive ratio multiplied by each locked multiplier",
        "Notebook 15 validation leader remains fixed; test results cannot reselect",
        "new pooled Version 2 validation F1 threshold applied unchanged; secondary",
        "quarterly top 1%, 5%, 10%, 20%; distinct failed-bank capture primary; positive-bank-quarter capture secondary",
        "1,000 percentile replicates; Version 1 CERT-cluster procedure and seed 20260816; common resamples across sources",
        "verified CERT transitions map to one audited bootstrap identity; IDRSSD remains the panel merge key",
        "equity_to_assets_lag1 with identical sample, ranking, capacity, measures and bootstrap",
        "secondary diagnostic only; class-weighted scores are not assumed calibrated",
        "realised source_report_date-to-failure days; longer than Version 1 although target remains (t,t+365]",
        "prohibited: no predictor, preprocessing, weight, hyperparameter, threshold, model or calibration relock after test access",
    ],
})

existing_complete_run = False
if RUN_MARKER_PATH.is_file():
    with open(RUN_MARKER_PATH, "r", encoding="utf-8") as handle:
        existing_marker = json.load(handle)
    existing_complete_run = existing_marker.get("status") == "COMPLETE"
    if existing_complete_run and os.environ.get("ALLOW_V2_FINAL_TEST_REPRODUCIBILITY_RERUN", "0") != "1":
        raise RuntimeError(
            "Notebook 16 has already completed. Use the saved outputs. An explicitly authorised "
            "reproducibility rerun requires ALLOW_V2_FINAL_TEST_REPRODUCIBILITY_RERUN=1."
        )

software_versions = {
    "python": os.sys.version.split()[0], "numpy": np.__version__, "pandas": pd.__version__,
    "scipy": scipy.__version__, "scikit_learn": sklearn.__version__, "xgboost": xgb.__version__,
}
cell_01_checks = {
    "Analysis is the Version 2 lagged-bank main design": ANALYSIS_VERSION == "v2_bank_lag1_main",
    "Bank and macro predictors remain fixed at t-1": TIMING_POLICY.get("bank_predictor_source") == "exact_t_minus_1" and TIMING_POLICY.get("macro_predictor_source") == "t_minus_1",
    "The target remains anchored at t over 365 days": TIMING_POLICY.get("target_window") == "(t, t_plus_365_days]" and TARGET_HORIZON_DAYS == 365,
    "All thirteen locked prerequisite files exist": len(required_paths) == 13 and all(path.is_file() for path in required_paths),
    "Capacity fractions are exactly 1%, 5%, 10% and 20%": TOP_RISK_PERCENTAGES == (0.01, 0.05, 0.10, 0.20),
    "Version 1 bootstrap seed and 1,000-replicate procedure are fixed": BOOTSTRAP_SEED == 20260816 and BOOTSTRAP_REPLICATES == 1000,
    "Notebook 16 writes only inside the Version 2 root": is_relative_to(NB16_OUTPUT_DIR, V2_ROOT),
    "No archived Version 1 path is loaded": all("version 1" not in str(path).lower() for path in required_paths),
    "A completed final-test run is not being overwritten": not existing_complete_run,
}
notebook_16_cell_01_gate = pd.DataFrame({
    "check": cell_01_checks.keys(),
    "result": ["PASS" if value else "FAIL" for value in cell_01_checks.values()],
    "passed": cell_01_checks.values(),
})
evaluation_contract.to_csv(NB16_OUTPUT_DIR / "cell_01_final_evaluation_contract.csv", index=False)
pd.DataFrame([software_versions]).to_csv(NB16_OUTPUT_DIR / "cell_01_software_versions.csv", index=False)
notebook_16_cell_01_gate.to_csv(NB16_OUTPUT_DIR / "cell_01_setup_gate.csv", index=False)

print("Notebook 16 Cell 1 validation gate:")
display(notebook_16_cell_01_gate)
print("\nOne-time Version 2 evaluation contract:")
with pd.option_context("display.max_colwidth", None):
    display(evaluation_contract)
failed = [name for name, passed in cell_01_checks.items() if not passed]
if failed:
    raise AssertionError("Cell 1 blocked: " + "; ".join(failed))
print("\nCell 1 completed successfully.")
print("Notebook 09–10 handoff root:", DATA_OUTPUT_ROOT)
print("Notebook 15 lock root:", LOCK_OUTPUT_ROOT)
print("Notebook 16 output directory:", NB16_OUTPUT_DIR)

In [ ]:
# Re-read Notebook 9, 10 and 15 locks; derive the final training, purge and test roles

if not notebook_16_cell_01_gate["passed"].all():
    raise AssertionError("Cell 2 blocked because Cell 1 did not pass.")

nb09_gate = pd.read_csv(NB09_FINAL_GATE_PATH)
nb10_gate = pd.read_csv(NB10_FINAL_GATE_PATH)
nb15_gate = pd.read_csv(NB15_FINAL_GATE_PATH)
with open(MODEL_LOCK_PATH, "r", encoding="utf-8") as handle:
    model_lock_manifest = json.load(handle)
validation_mean_comparison = pd.read_csv(VALIDATION_COMPARISON_PATH)
validation_source_manifest = pd.read_csv(VALIDATION_SOURCE_MANIFEST_PATH)
membership = pd.read_parquet(MEMBERSHIP_PATH)
period_summary = pd.read_csv(PERIOD_SUMMARY_PATH, parse_dates=["start", "end"])

membership["report_date"] = pd.to_datetime(membership["report_date"], errors="raise").dt.normalize()
membership["cert"] = pd.to_numeric(membership["cert"], errors="raise").astype("int64")
membership["idrssd"] = pd.to_numeric(membership["idrssd"], errors="raise").astype("int64")
membership[TARGET] = pd.to_numeric(membership[TARGET], errors="raise").astype("int8")

locked_models = model_lock_manifest["locked_models"]
locked_model_ids = {record["model_id"] for record in locked_models}
LEADER_MODEL_ID = str(model_lock_manifest["selection"]["leader_model_id"])
role_summary = period_summary.loc[
    period_summary["period"].isin(["final_train", "final_purge", "final_test"])
].copy()
role_summary["role"] = role_summary["period"].map({
    "final_train": "training", "final_purge": "purge", "final_test": "test"
})
role_lookup = role_summary.set_index("role")

final_train_start = quarter_end(FINAL_TEST_DESIGN["train_start"])
final_train_end = quarter_end(FINAL_TEST_DESIGN["train_end"])
final_purge_start = quarter_end(FINAL_TEST_DESIGN["purge_start"])
final_purge_end = quarter_end(FINAL_TEST_DESIGN["purge_end"])
final_test_start = quarter_end(FINAL_TEST_DESIGN["test_start"])
final_test_end = quarter_end(FINAL_TEST_DESIGN["test_end"])

lock_sha256 = sha256_file(MODEL_LOCK_PATH)
membership_key_sha256 = hashlib.sha256(
    pd.util.hash_pandas_object(
        membership[["panel_row_id", *PANEL_KEYS, "final_role"]], index=False
    ).values.tobytes()
).hexdigest()
RUN_SIGNATURE = hashlib.sha256(
    (
        lock_sha256 + membership_key_sha256
        + hashlib.sha256(
            json.dumps(input_hashes_before, sort_keys=True).encode("utf-8")
        ).hexdigest()
        + json.dumps(software_versions, sort_keys=True)
        + str(RANDOM_SEED) + str(BOOTSTRAP_SEED)
    ).encode("utf-8")
).hexdigest()

marker_payload = {
    "status": "IN_PROGRESS", "analysis_version": ANALYSIS_VERSION,
    "run_signature": RUN_SIGNATURE, "started_utc": datetime.now(timezone.utc).isoformat(),
    "model_lock_sha256": lock_sha256, "membership_key_sha256": membership_key_sha256,
    "test_based_reselection_performed": False,
}
strict_json_dump(marker_payload, RUN_MARKER_PATH)

model_lock_rows = []
for record in locked_models:
    family = str(record["model_family"])
    specification = str(record["specification"])
    predictors = list(record["predictors_in_order"])
    model_lock_rows.append({
        "model_id": record["model_id"], "model_family": family,
        "specification": specification, "config_id": str(record["config_id"]),
        "predictor_count": len(predictors),
        "expected_predictor_count": 19 if specification == "bank_only" else 23,
        "history_indicator_present": HISTORY_INDICATOR in predictors,
        "pooled_validation_f1_threshold": float(record["pooled_validation_f1_threshold"]),
        "positive_weight_multiplier": float(record["hyperparameters"]["positive_weight_multiplier"]),
        "fold_preprocessing_reused": bool(record["preprocessing"]["fold_preprocessing_objects_reused_in_final_fit"]),
        "selection_status": str(record["selection_status"]),
    })
model_lock_audit = pd.DataFrame(model_lock_rows)

cell_02_checks = {
    "Notebook 9, 10 and 15 final gates are fully passed": passed_values(nb09_gate["passed"]).all() and passed_values(nb10_gate["passed"]).all() and passed_values(nb15_gate["passed"]).all(),
    "The membership handoff has unique panel rows and bank-quarter keys": not membership["panel_row_id"].duplicated().any() and not membership.duplicated(PANEL_KEYS).any(),
    "Every row has a final training, purge or test role": membership["final_role"].isin(["training", "purge", "test"]).all(),
    "The final calendar is 2001Q2–2018Q4, 2019 purge, 2020Q1–2025Q2 test": final_train_start == pd.Timestamp("2001-06-30") and final_train_end == pd.Timestamp("2018-12-31") and final_purge_start == pd.Timestamp("2019-03-31") and final_purge_end == pd.Timestamp("2019-12-31") and final_test_start == pd.Timestamp("2020-03-31") and final_test_end == pd.Timestamp("2025-06-30"),
    "Final role counts reconcile to the full membership handoff": int(role_summary["observations"].sum()) == len(membership) and all(int(role_lookup.loc[role, "observations"]) == int(membership["final_role"].eq(role).sum()) for role in ["training", "purge", "test"]),
    "The final test contains 92,229 rows, 40 positives, 13 failed banks and 22 quarters": int(role_lookup.loc["test", "observations"]) == EXPECTED_TEST_ROWS and int(role_lookup.loc["test", "positive_bank_quarters"]) == EXPECTED_TEST_POSITIVES and int(role_lookup.loc["test", "distinct_failed_banks"]) == EXPECTED_TEST_FAILED_BANKS and int(role_lookup.loc["test", "report_quarters"]) == EXPECTED_TEST_QUARTERS,
    "Exactly six locked model specifications are present": locked_model_ids == EXPECTED_MODEL_IDS and len(locked_models) == 6,
    "The validation leader remains Random Forest bank plus macro": LEADER_MODEL_ID == "random_forest__bank_plus_macro" and validation_mean_comparison.loc[validation_mean_comparison["selection_status"].eq("LOCKED VALIDATION LEADER"), "model_id"].tolist() == [LEADER_MODEL_ID],
    "Every lock has the exact predictor count, history indicator and bounded threshold": model_lock_audit["predictor_count"].eq(model_lock_audit["expected_predictor_count"]).all() and model_lock_audit["history_indicator_present"].all() and model_lock_audit["pooled_validation_f1_threshold"].between(0, 1).all(),
    "Fold preprocessing objects are barred from final fitting": not model_lock_audit["fold_preprocessing_reused"].any(),
    "Notebook 15 confirms no final-test information entered selection": model_lock_manifest["selection"]["final_test_information_used"] is False and model_lock_manifest["final_test_access_audit"]["final_test_rows_loaded"] == 0,
}
notebook_16_cell_02_gate = pd.DataFrame({
    "check": cell_02_checks.keys(),
    "result": ["PASS" if value else "FAIL" for value in cell_02_checks.values()],
    "passed": cell_02_checks.values(),
})
role_summary.to_csv(NB16_OUTPUT_DIR / "cell_02_final_role_summary.csv", index=False)
model_lock_audit.to_csv(NB16_OUTPUT_DIR / "cell_02_model_lock_audit.csv", index=False)
notebook_16_cell_02_gate.to_csv(NB16_OUTPUT_DIR / "cell_02_lock_and_role_gate.csv", index=False)

print("Notebook 16 Cell 2 validation gate:")
display(notebook_16_cell_02_gate)
print("\nFinal-stage role summary after exact-lag exclusions:")
display(role_summary)
print("\nSix immutable model locks:")
display(model_lock_audit)
failed = [name for name, passed in cell_02_checks.items() if not passed]
if failed:
    raise AssertionError("Cell 2 blocked: " + "; ".join(failed))
print("\nCell 2 completed successfully. Run signature:", RUN_SIGNATURE[:16])

In [ ]:
# Load only final-training and final-test panel values; audit CERT identities and realised horizons

if not notebook_16_cell_02_gate["passed"].all():
    raise AssertionError("Cell 3 blocked because Cell 2 did not pass.")

all_predictors = []
for record in locked_models:
    for predictor in record["predictors_in_order"]:
        if predictor not in all_predictors:
            all_predictors.append(predictor)
panel_columns = PANEL_KEYS + all_predictors + [TARGET]

# Parquet filters ensure 2019 predictor values and outcomes are never loaded into memory.
train_panel = pd.read_parquet(
    FINAL_PANEL_PATH, columns=panel_columns,
    filters=[("report_date", "<=", final_train_end)],
)
test_panel = pd.read_parquet(
    FINAL_PANEL_PATH, columns=panel_columns,
    filters=[("report_date", ">=", final_test_start)],
)
for frame in [train_panel, test_panel]:
    frame["report_date"] = pd.to_datetime(frame["report_date"], errors="raise").dt.normalize()
    frame["cert"] = pd.to_numeric(frame["cert"], errors="raise").astype("int64")
    frame["idrssd"] = pd.to_numeric(frame["idrssd"], errors="raise").astype("int64")
    frame[TARGET] = pd.to_numeric(frame[TARGET], errors="raise").astype("int8")

membership_columns = ["panel_row_id", *PANEL_KEYS, TARGET, "final_role"]
train_roles = membership.loc[membership["final_role"].eq("training"), membership_columns]
test_roles = membership.loc[membership["final_role"].eq("test"), membership_columns]

def attach_role(panel_frame, role_frame):
    merged = panel_frame.merge(
        role_frame, on=PANEL_KEYS, how="inner", suffixes=("_panel", "_membership"),
        validate="one_to_one",
    )
    if not merged[f"{TARGET}_panel"].eq(merged[f"{TARGET}_membership"]).all():
        raise AssertionError("Notebook 9 and Notebook 10 targets disagree.")
    merged[TARGET] = merged.pop(f"{TARGET}_panel").astype("int8")
    merged.drop(columns=[f"{TARGET}_membership"], inplace=True)
    merged.sort_values("panel_row_id", kind="mergesort", inplace=True)
    merged.reset_index(drop=True, inplace=True)
    return merged

train_data = attach_role(train_panel, train_roles)
test_data = attach_role(test_panel, test_roles)


def audited_cert_components(allowlist):
    parent = {}

    def find(value):
        parent.setdefault(value, value)
        while parent[value] != value:
            parent[value] = parent[parent[value]]
            value = parent[value]
        return value

    def union(left, right):
        left_root, right_root = find(left), find(right)
        if left_root != right_root:
            canonical = min(left_root, right_root)
            other = max(left_root, right_root)
            parent[other] = canonical

    for row in allowlist.itertuples(index=False):
        union(int(row.cert_source), int(row.cert_reference))
    components = {}
    for value in list(parent):
        components.setdefault(find(value), []).append(value)
    identity_map = {}
    audit_rows = []
    for members in components.values():
        members = sorted(set(int(value) for value in members))
        canonical = min(members)
        for value in members:
            identity_map[value] = canonical
            audit_rows.append({
                "original_cert": value, "bootstrap_identity": canonical,
                "component_members": ",".join(map(str, members)),
                "mapping_rule": "minimum CERT in audited connected transition component",
            })
    return identity_map, pd.DataFrame(audit_rows)


cert_allowlist = pd.read_csv(CERT_ALLOWLIST_PATH)
cert_exclusions = pd.read_csv(CERT_EXCLUSION_PATH)
cert_identity_map, cert_bootstrap_identity_audit = audited_cert_components(cert_allowlist)
for frame in [train_data, test_data]:
    frame["bootstrap_identity"] = frame["cert"].map(cert_identity_map).fillna(frame["cert"]).astype("int64")

allowlist_identity_check = cert_allowlist.assign(
    source_identity=cert_allowlist["cert_source"].map(cert_identity_map),
    reference_identity=cert_allowlist["cert_reference"].map(cert_identity_map),
)
cert_bootstrap_identity_audit.to_csv(
    NB16_OUTPUT_DIR / "v2_cert_to_bootstrap_identity_audit.csv", index=False
)

horizon_audit = pd.read_parquet(HORIZON_AUDIT_PATH)
for column in ["report_date", "source_report_date", "fail_date"]:
    horizon_audit[column] = pd.to_datetime(horizon_audit[column], errors="raise").dt.normalize()
horizon_audit["analysis_scope"] = np.select(
    [
        horizon_audit["report_date"].between(final_train_start, final_train_end),
        horizon_audit["report_date"].between(final_test_start, final_test_end),
    ],
    ["final_training", "final_test"],
    default="excluded_2019_purge",
)
horizon_summary_rows = []
for scope in ["final_training", "final_test", "combined_train_and_test"]:
    if scope == "combined_train_and_test":
        scoped = horizon_audit.loc[horizon_audit["analysis_scope"].isin(["final_training", "final_test"])]
    else:
        scoped = horizon_audit.loc[horizon_audit["analysis_scope"].eq(scope)]
    values = scoped["predictor_to_failure_days"].astype("float64")
    horizon_summary_rows.append({
        "scope": scope, "positive_bank_quarters": int(len(scoped)),
        "distinct_failed_bootstrap_identities": int(scoped["cert_reference"].map(cert_identity_map).fillna(scoped["cert_reference"]).nunique()),
        "minimum_days": float(values.min()), "fifth_percentile_days": float(values.quantile(0.05)),
        "median_days": float(values.median()), "maximum_days": float(values.max()),
        "outcome_window": "(t, t+365]",
        "interpretation": "Longer than Version 1 because the bank predictor date is t-1.",
    })
predictor_horizon_summary = pd.DataFrame(horizon_summary_rows)
predictor_horizon_summary.to_csv(
    NB16_OUTPUT_DIR / "realised_predictor_date_to_failure_horizon.csv", index=False
)

training_positives = int(train_data[TARGET].sum())
training_negatives = int(len(train_data) - training_positives)
FINAL_TRAINING_IMBALANCE_RATIO = training_negatives / training_positives
sample_summary = pd.DataFrame({
    "sample": ["final_training", "purge_not_loaded", "final_test"],
    "reference_period": [
        f'{FINAL_TEST_DESIGN["train_start"]}–{FINAL_TEST_DESIGN["train_end"]}',
        f'{FINAL_TEST_DESIGN["purge_start"]}–{FINAL_TEST_DESIGN["purge_end"]}',
        f'{FINAL_TEST_DESIGN["test_start"]}–{FINAL_TEST_DESIGN["test_end"]}',
    ],
    "observations": [len(train_data), int(role_lookup.loc["purge", "observations"]), len(test_data)],
    "positive_bank_quarters": [training_positives, np.nan, int(test_data[TARGET].sum())],
    "distinct_raw_certs": [train_data.loc[train_data[TARGET].eq(1), "cert"].nunique(), np.nan, test_data.loc[test_data[TARGET].eq(1), "cert"].nunique()],
    "distinct_audited_bootstrap_identities": [train_data.loc[train_data[TARGET].eq(1), "bootstrap_identity"].nunique(), np.nan, test_data.loc[test_data[TARGET].eq(1), "bootstrap_identity"].nunique()],
    "report_quarters": [train_data["report_date"].nunique(), 4, test_data["report_date"].nunique()],
    "panel_values_loaded": [True, False, True],
})

cell_03_checks = {
    "Training rows are exactly the Notebook 10 final-training membership": len(train_data) == int(role_lookup.loc["training", "observations"]) and train_data["final_role"].eq("training").all(),
    "Test rows are exactly the Notebook 10 final-test membership": len(test_data) == EXPECTED_TEST_ROWS and test_data["final_role"].eq("test").all(),
    "No 2019 panel value was loaded": train_panel["report_date"].max() == final_train_end and test_panel["report_date"].min() == final_test_start,
    "Training begins at 2001Q2 and test covers exactly 22 quarters": train_data["report_date"].min() == final_train_start and test_data["report_date"].nunique() == EXPECTED_TEST_QUARTERS,
    "The test has 40 positives and 13 audited failed-bank identities": int(test_data[TARGET].sum()) == EXPECTED_TEST_POSITIVES and test_data.loc[test_data[TARGET].eq(1), "bootstrap_identity"].nunique() == EXPECTED_TEST_FAILED_BANKS,
    "Every one of the 41 verified CERT transitions maps to one identity": len(cert_allowlist) == 41 and allowlist_identity_check["source_identity"].eq(allowlist_identity_check["reference_identity"]).all(),
    "No identifier-error exclusion was carried into the panel": len(cert_exclusions) == 0,
    "IDRSSD remains the unique panel merge key with report date": not train_data.duplicated(["report_date", "idrssd"]).any() and not test_data.duplicated(["report_date", "idrssd"]).any(),
    "All 23 locked predictors are present and no unsuffixed bank feature was substituted": len(all_predictors) == 23 and set(all_predictors) <= set(train_data.columns) and all(name.endswith("_lag1") for name in all_predictors if name not in {"real_gdp_growth_lag1", "unemployment_rate_lag1", "effective_federal_funds_rate_lag1", "house_price_growth_lag1"}),
    "The realised source-date horizon reconciles and is longer than Version 1": horizon_audit["predictor_to_failure_days"].eq(horizon_audit["days_to_failure"] + (horizon_audit["report_date"] - horizon_audit["source_report_date"]).dt.days).all() and horizon_audit["predictor_to_failure_days"].gt(horizon_audit["days_to_failure"]).all(),
    "The final-test horizon audit contains all 40 positive bank-quarters": int(predictor_horizon_summary.loc[predictor_horizon_summary["scope"].eq("final_test"), "positive_bank_quarters"].iloc[0]) == EXPECTED_TEST_POSITIVES,
    "The final-training class ratio uses training labels only": np.isfinite(FINAL_TRAINING_IMBALANCE_RATIO) and FINAL_TRAINING_IMBALANCE_RATIO > 1,
}
notebook_16_cell_03_gate = pd.DataFrame({
    "check": cell_03_checks.keys(),
    "result": ["PASS" if value else "FAIL" for value in cell_03_checks.values()],
    "passed": cell_03_checks.values(),
})
sample_summary.to_csv(NB16_OUTPUT_DIR / "cell_03_final_samples.csv", index=False)
notebook_16_cell_03_gate.to_csv(NB16_OUTPUT_DIR / "cell_03_sample_identity_and_horizon_gate.csv", index=False)

print("Notebook 16 Cell 3 validation gate:")
display(notebook_16_cell_03_gate)
print("\nLoaded samples; 2019 panel values were not read:")
display(sample_summary)
print("\nRealised predictor-date-to-failure horizon:")
display(predictor_horizon_summary)
print("\nThe horizon is longer than Version 1 even though the outcome remains (t, t+365].")
failed = [name for name, passed in cell_03_checks.items() if not passed]
if failed:
    raise AssertionError("Cell 3 blocked: " + "; ".join(failed))
print("\nCell 3 completed successfully.")

In [ ]:
# Refit final-training preprocessing and all six locked models; score the test once

if not notebook_16_cell_03_gate["passed"].all():
    raise AssertionError("Cell 4 blocked because Cell 3 did not pass.")


def fit_final_preprocessor(training_frame, scoring_frame, predictors, standardise):
    raw_training = training_frame[predictors].astype("float64")
    raw_scoring = scoring_frame[predictors].astype("float64")
    if np.isinf(raw_training.to_numpy()).any() or np.isinf(raw_scoring.to_numpy()).any():
        raise AssertionError("Predictor matrix contains infinity.")
    medians = raw_training.median(axis=0, skipna=True)
    if medians.isna().any() or not np.isfinite(medians.to_numpy()).all():
        bad = medians.index[~np.isfinite(medians.to_numpy())].tolist()
        raise AssertionError("No finite final-training median for: " + ", ".join(bad))
    training_imputed = raw_training.fillna(medians)
    scoring_imputed = raw_scoring.fillna(medians)
    continuous = [name for name in predictors if name != HISTORY_INDICATOR]
    lower = training_imputed[continuous].quantile(WINSOR_LOWER_QUANTILE, interpolation="linear")
    upper = training_imputed[continuous].quantile(WINSOR_UPPER_QUANTILE, interpolation="linear")
    if (lower > upper).any():
        raise AssertionError("A fitted lower winsor bound exceeds its upper bound.")
    training_imputed[continuous] = training_imputed[continuous].clip(lower=lower, upper=upper, axis=1)
    scoring_imputed[continuous] = scoring_imputed[continuous].clip(lower=lower, upper=upper, axis=1)
    scaler = None
    if standardise:
        scaler = StandardScaler()
        x_training = scaler.fit_transform(training_imputed[predictors])
        x_scoring = scaler.transform(scoring_imputed[predictors])
    else:
        x_training = training_imputed[predictors].to_numpy(dtype="float32")
        x_scoring = scoring_imputed[predictors].to_numpy(dtype="float32")
    if not np.isfinite(x_training).all() or not np.isfinite(x_scoring).all():
        raise AssertionError("Final preprocessing produced a non-finite value.")
    parameters = {
        "analysis_version": ANALYSIS_VERSION, "run_signature": RUN_SIGNATURE,
        "predictors": list(predictors), "fit_scope": "eligible Version 2 reference rows 2001Q2–2018Q4 only",
        "test_data_used_to_fit_parameters": False, "purge_data_used": False,
        "imputation_strategy": "training-sample median",
        "imputation_medians": {name: float(medians[name]) for name in predictors},
        "winsor_lower_quantile": WINSOR_LOWER_QUANTILE,
        "winsor_upper_quantile": WINSOR_UPPER_QUANTILE,
        "winsor_quantile_interpolation": "linear",
        "winsor_excluded": [HISTORY_INDICATOR],
        "winsor_lower_bounds": {name: float(lower[name]) for name in continuous},
        "winsor_upper_bounds": {name: float(upper[name]) for name in continuous},
        "standardisation_applied": bool(standardise),
        "scaler_means": None if scaler is None else {name: float(value) for name, value in zip(predictors, scaler.mean_)},
        "scaler_scales": None if scaler is None else {name: float(value) for name, value in zip(predictors, scaler.scale_)},
    }
    return x_training, x_scoring, parameters


def build_locked_model(record, positive_weight):
    family = str(record["model_family"])
    hyper = dict(record["hyperparameters"])
    if family == "logistic_regression":
        return LogisticRegression(
            penalty=str(hyper["regularisation_penalty"]), C=float(hyper["C"]),
            class_weight={0: 1.0, 1: positive_weight}, solver=str(hyper["solver"]),
            max_iter=int(record["implementation"]["max_iter"]), random_state=RANDOM_SEED,
        )
    if family == "random_forest":
        max_depth = hyper.get("max_depth")
        if max_depth is not None:
            max_depth = int(max_depth)
        return RandomForestClassifier(
            n_estimators=int(hyper["n_estimators"]), max_depth=max_depth,
            min_samples_leaf=int(hyper["min_samples_leaf"]), max_features=hyper["max_features"],
            class_weight={0: 1.0, 1: positive_weight}, criterion="gini", bootstrap=True,
            max_samples=float(hyper["max_samples"]), random_state=RANDOM_SEED, n_jobs=-1,
        )
    if family == "xgboost":
        return XGBClassifier(
            n_estimators=int(hyper["n_estimators"]), max_depth=int(hyper["max_depth"]),
            learning_rate=float(hyper["learning_rate"]), min_child_weight=float(hyper["min_child_weight"]),
            subsample=float(hyper["subsample"]), colsample_bytree=float(hyper["colsample_bytree"]),
            gamma=float(hyper["gamma"]), reg_lambda=float(hyper["reg_lambda"]),
            reg_alpha=float(hyper["reg_alpha"]), scale_pos_weight=float(positive_weight),
            objective="binary:logistic", eval_metric="logloss", tree_method=str(hyper["tree_method"]),
            importance_type="gain", random_state=RANDOM_SEED, n_jobs=-1,
            verbosity=0, validate_parameters=True,
        )
    raise ValueError(f"Unsupported model family: {family}")


def probability_metrics(y_true, probability, threshold):
    prediction = probability >= threshold
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return {
        "roc_auc": float(roc_auc_score(y_true, probability)),
        "pr_auc": float(average_precision_score(y_true, probability)),
        "locked_threshold": float(threshold),
        "observations_flagged": int(prediction.sum()),
        "percentage_flagged": float(prediction.mean()),
        "true_positive": int(tp), "false_positive": int(fp),
        "true_negative": int(tn), "false_negative": int(fn),
        "recall": float(recall_score(y_true, prediction, zero_division=0)),
        "precision": float(precision_score(y_true, prediction, zero_division=0)),
        "f1": float(f1_score(y_true, prediction, zero_division=0)),
    }


def extract_feature_effects(model, record):
    predictors = list(record["predictors_in_order"])
    family = str(record["model_family"])
    if family == "logistic_regression":
        result = pd.DataFrame({"predictor": predictors, "effect_value": model.coef_[0]})
        result["effect_measure"] = "standardised_coefficient"
        result = pd.concat([result, pd.DataFrame({
            "predictor": ["__intercept__"], "effect_value": [float(model.intercept_[0])],
            "effect_measure": ["standardised_coefficient"],
        })], ignore_index=True)
    else:
        result = pd.DataFrame({"predictor": predictors, "effect_value": model.feature_importances_})
        result["effect_measure"] = "impurity_importance" if family == "random_forest" else "gain_importance"
    result.insert(0, "model_id", record["model_id"])
    return result


y_training = train_data[TARGET].to_numpy(dtype="int8")
y_test = test_data[TARGET].to_numpy(dtype="int8")
test_prediction_frames = {}
fit_rows, metric_rows, feature_parts, artifact_rows = [], [], [], []

for job_number, record in enumerate(sorted(locked_models, key=lambda item: item["model_id"]), start=1):
    model_id = str(record["model_id"])
    family = str(record["model_family"])
    predictors = list(record["predictors_in_order"])
    threshold = float(record["pooled_validation_f1_threshold"])
    multiplier = float(record["hyperparameters"]["positive_weight_multiplier"])
    positive_weight = float(FINAL_TRAINING_IMBALANCE_RATIO * multiplier)
    prediction_path = PREDICTION_DIR / f"{model_id}__v2_final_test_predictions.parquet"
    model_path = MODEL_DIR / f"{model_id}.joblib"
    preprocessing_path = PREPROCESSING_DIR / f"{model_id}__final_training_preprocessing.json"
    feature_path = MODEL_DIR / f"{model_id}__feature_effects.csv"
    checkpoint_path = CHECKPOINT_DIR / f"{model_id}__complete.json"
    resumed = False
    fit_seconds = 0.0

    complete_artifacts = all(path.is_file() for path in [prediction_path, model_path, preprocessing_path, feature_path, checkpoint_path])
    if complete_artifacts:
        with open(checkpoint_path, "r", encoding="utf-8") as handle:
            checkpoint = json.load(handle)
        if checkpoint.get("run_signature") != RUN_SIGNATURE or checkpoint.get("model_id") != model_id:
            raise AssertionError(f"Stale or mismatched final-fit checkpoint: {checkpoint_path}")
        predictions = pd.read_parquet(prediction_path)
        feature_effects = pd.read_csv(feature_path)
        fit_seconds = float(checkpoint.get("fit_seconds", 0.0))
        resumed = True
        print(f"[{job_number}/6] RESUMED same-signature checkpoint: {model_id}")
    else:
        standardise = family == "logistic_regression"
        x_training, x_scoring, preprocessing_parameters = fit_final_preprocessor(
            train_data, test_data, predictors, standardise
        )
        preprocessing_parameters.update({
            "model_id": model_id, "model_family": family,
            "final_training_observations": int(len(train_data)),
            "final_training_positives": int(y_training.sum()),
            "final_training_imbalance_ratio": float(FINAL_TRAINING_IMBALANCE_RATIO),
            "positive_weight_multiplier": multiplier,
            "positive_class_weight_or_scale_pos_weight": positive_weight,
        })
        model = build_locked_model(record, positive_weight)
        fit_start = time.perf_counter()
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            model.fit(x_training, y_training)
        convergence_warning = any(issubclass(item.category, ConvergenceWarning) for item in caught)
        probability = model.predict_proba(x_scoring)[:, 1].astype("float64")
        fit_seconds = time.perf_counter() - fit_start
        predictions = test_data[["panel_row_id", *PANEL_KEYS, "bootstrap_identity", TARGET]].copy()
        predictions["predicted_probability"] = probability
        predictions["analysis_version"] = ANALYSIS_VERSION
        predictions["model_id"] = model_id
        predictions["model_family"] = family
        predictions["specification"] = record["specification"]
        predictions["config_id"] = str(record["config_id"])
        predictions["locked_threshold"] = threshold
        predictions["test_used_to_fit_preprocessing"] = False
        feature_effects = extract_feature_effects(model, record)
        predictions.to_parquet(prediction_path, index=False, compression="snappy")
        feature_effects.to_csv(feature_path, index=False)
        strict_json_dump(preprocessing_parameters, preprocessing_path)
        joblib.dump(model, model_path, compress=3)
        strict_json_dump({
            "status": "COMPLETE", "run_signature": RUN_SIGNATURE,
            "model_id": model_id, "fit_seconds": fit_seconds,
            "test_observations": int(len(predictions)),
            "prediction_sha256": sha256_file(prediction_path),
            "test_based_reselection_performed": False,
        }, checkpoint_path)
        print(f"[{job_number}/6] FIT {model_id} | {fit_seconds / 60:.2f} minutes")

    predictions["report_date"] = pd.to_datetime(predictions["report_date"], errors="raise").dt.normalize()
    predictions.sort_values("panel_row_id", kind="mergesort", inplace=True)
    predictions.reset_index(drop=True, inplace=True)
    expected_rows = test_data[["panel_row_id", *PANEL_KEYS, "bootstrap_identity", TARGET]].reset_index(drop=True)
    if not predictions[["panel_row_id", *PANEL_KEYS, "bootstrap_identity", TARGET]].equals(expected_rows):
        raise AssertionError(f"Saved predictions do not match the locked test rows for {model_id}.")
    probability = predictions["predicted_probability"].to_numpy(dtype="float64")
    metrics = probability_metrics(y_test, probability, threshold)
    test_prediction_frames[model_id] = predictions
    fit_rows.append({
        "model_id": model_id, "model_family": family,
        "specification": record["specification"], "config_id": str(record["config_id"]),
        "training_observations": int(len(train_data)), "training_positives": int(y_training.sum()),
        "training_imbalance_ratio": float(FINAL_TRAINING_IMBALANCE_RATIO),
        "positive_weight_multiplier": multiplier,
        "positive_class_weight_or_scale_pos_weight": positive_weight,
        "fit_seconds": fit_seconds, "resumed_from_same_signature_checkpoint": resumed,
        "convergence_warning": bool(convergence_warning) if not resumed and family == "logistic_regression" else False,
        "prediction_file": str(prediction_path.relative_to(NB16_OUTPUT_DIR)),
        "model_file": str(model_path.relative_to(NB16_OUTPUT_DIR)),
        "preprocessing_file": str(preprocessing_path.relative_to(NB16_OUTPUT_DIR)),
    })
    metric_rows.append({
        "model_id": model_id, "model_family": family,
        "specification": record["specification"],
        "validation_selected_overall_leader": model_id == LEADER_MODEL_ID,
        "test_observations": int(len(test_data)),
        "positive_bank_quarters": int(y_test.sum()),
        "distinct_failed_banks": int(test_data.loc[test_data[TARGET].eq(1), "bootstrap_identity"].nunique()),
        "test_prevalence": float(y_test.mean()),
        "pr_auc_divided_by_prevalence": metrics["pr_auc"] / float(y_test.mean()),
        **metrics,
        "threshold_source": "pooled Version 2 out-of-time validation predictions; applied unchanged",
        "threshold_result_priority": "secondary",
        "test_used_for_selection": False,
    })
    feature_parts.append(feature_effects)
    artifact_rows.append({
        "model_id": model_id, "model_exists": model_path.is_file(),
        "prediction_exists": prediction_path.is_file(),
        "preprocessing_exists": preprocessing_path.is_file(),
        "feature_effects_exists": feature_path.is_file(),
        "checkpoint_exists": checkpoint_path.is_file(),
    })

final_fit_register = pd.DataFrame(fit_rows)
final_test_point_metrics = pd.DataFrame(metric_rows)
final_model_feature_effects = pd.concat(feature_parts, ignore_index=True)
model_artifact_audit = pd.DataFrame(artifact_rows)
final_fit_register.to_csv(NB16_OUTPUT_DIR / "cell_04_final_fit_register.csv", index=False)
final_test_point_metrics.to_csv(NB16_OUTPUT_DIR / "v2_final_test_point_metrics.csv", index=False)
final_model_feature_effects.to_csv(NB16_OUTPUT_DIR / "v2_final_model_feature_effects.csv", index=False)
model_artifact_audit.to_csv(NB16_OUTPUT_DIR / "cell_04_model_artifact_audit.csv", index=False)

cell_04_checks = {
    "Exactly six locked models were fitted or validly recovered": len(final_fit_register) == 6 and set(final_fit_register["model_id"]) == EXPECTED_MODEL_IDS,
    "Every fit uses the full final-training sample and no purge row": final_fit_register["training_observations"].eq(len(train_data)).all() and train_data["report_date"].max() == final_train_end,
    "Every positive weight equals the training ratio times its locked multiplier": np.allclose(final_fit_register["positive_class_weight_or_scale_pos_weight"], final_fit_register["training_imbalance_ratio"] * final_fit_register["positive_weight_multiplier"]),
    "Each model produced one finite probability for every locked test row": len(test_prediction_frames) == 6 and all(len(frame) == EXPECTED_TEST_ROWS and np.isfinite(frame["predicted_probability"]).all() and frame["predicted_probability"].between(0, 1).all() for frame in test_prediction_frames.values()),
    "All test predictions cover the same 40 positives and 13 failed-bank identities": final_test_point_metrics["positive_bank_quarters"].eq(EXPECTED_TEST_POSITIVES).all() and final_test_point_metrics["distinct_failed_banks"].eq(EXPECTED_TEST_FAILED_BANKS).all(),
    "All six new pooled Version 2 thresholds were applied unchanged": np.allclose(sorted(final_test_point_metrics["locked_threshold"]), sorted(float(record["pooled_validation_f1_threshold"]) for record in locked_models)),
    "All model and audit artifacts exist": model_artifact_audit.drop(columns="model_id").all().all(),
    "The validation leader remains labelled once and no test row selects a model": final_test_point_metrics["validation_selected_overall_leader"].sum() == 1 and not final_test_point_metrics["test_used_for_selection"].any(),
}
notebook_16_cell_04_gate = pd.DataFrame({
    "check": cell_04_checks.keys(),
    "result": ["PASS" if value else "FAIL" for value in cell_04_checks.values()],
    "passed": cell_04_checks.values(),
})
notebook_16_cell_04_gate.to_csv(NB16_OUTPUT_DIR / "cell_04_final_fit_gate.csv", index=False)

print("Notebook 16 Cell 4 validation gate:")
display(notebook_16_cell_04_gate)
print("\nOne-time Version 2 final-test point estimates; the validation leader is not reselected:")
display(final_test_point_metrics[[
    "model_id", "validation_selected_overall_leader", "roc_auc", "pr_auc",
    "pr_auc_divided_by_prevalence", "locked_threshold", "percentage_flagged",
    "recall", "precision", "f1",
]])
failed = [name for name, passed in cell_04_checks.items() if not passed]
if failed:
    raise AssertionError("Cell 4 blocked: " + "; ".join(failed))
print("\nCell 4 completed successfully. Test-based reselection was not performed.")

In [ ]:
# Apply quarterly capacities, the unchanged V2 thresholds and the lagged low-equity baseline

if not notebook_16_cell_04_gate["passed"].all():
    raise AssertionError("Cell 5 blocked because Cell 4 did not pass.")


def build_quarterly_rankings(frame, source_id, source_type):
    base = frame[["panel_row_id", *PANEL_KEYS, "bootstrap_identity", TARGET, "risk_score"]].copy().reset_index(drop=True)
    base["_row_position"] = np.arange(len(base), dtype="int64")
    ranked = base.sort_values(
        ["report_date", "risk_score", "cert", "idrssd", "panel_row_id"],
        ascending=[True, False, True, True, True], kind="mergesort",
    ).copy()
    ranked["quarter_rank"] = ranked.groupby("report_date").cumcount() + 1
    ranked["active_banks_in_quarter"] = ranked.groupby("report_date")["panel_row_id"].transform("size")
    membership_masks, quarterly_rows, summary_rows = {}, [], []
    total_failed_banks = int(base.loc[base[TARGET].eq(1), "bootstrap_identity"].nunique())
    total_positive_quarters = int(base[TARGET].sum())
    prevalence = float(base[TARGET].mean())
    quarter_count = int(base["report_date"].nunique())
    for capacity in TOP_RISK_PERCENTAGES:
        group_size = np.maximum(1, np.ceil(ranked["active_banks_in_quarter"] * capacity)).astype(int)
        ranked["flagged"] = ranked["quarter_rank"].le(group_size)
        flag_mask = np.zeros(len(base), dtype=bool)
        flag_mask[ranked["_row_position"].to_numpy()] = ranked["flagged"].to_numpy()
        membership_masks[capacity] = flag_mask
        for report_date, quarter in ranked.groupby("report_date", sort=True):
            flagged = quarter.loc[quarter["flagged"]]
            quarterly_rows.append({
                "source_id": source_id, "source_type": source_type,
                "report_date": report_date, "top_risk_percentage": capacity,
                "active_banks": int(len(quarter)), "observations_flagged": int(len(flagged)),
                "percentage_flagged": float(len(flagged) / len(quarter)),
                "positive_bank_quarters": int(quarter[TARGET].sum()),
                "positive_bank_quarters_captured": int(flagged[TARGET].sum()),
                "distinct_failed_banks": int(quarter.loc[quarter[TARGET].eq(1), "bootstrap_identity"].nunique()),
                "distinct_failed_banks_captured": int(flagged.loc[flagged[TARGET].eq(1), "bootstrap_identity"].nunique()),
            })
        flagged = base.loc[flag_mask]
        positives_captured = int(flagged[TARGET].sum())
        banks_captured = int(flagged.loc[flagged[TARGET].eq(1), "bootstrap_identity"].nunique())
        precision = positives_captured / len(flagged)
        summary_rows.append({
            "source_id": source_id, "source_type": source_type,
            "top_risk_percentage": capacity, "test_observations": int(len(base)),
            "test_quarters": quarter_count, "test_prevalence": prevalence,
            "observations_flagged_total": int(len(flagged)),
            "observations_flagged_mean_per_quarter": float(len(flagged) / quarter_count),
            "percentage_flagged": float(len(flagged) / len(base)),
            "precision_in_top_group": float(precision),
            "lift_over_test_prevalence": float(precision / prevalence),
            "distinct_failed_banks_total": total_failed_banks,
            "distinct_failed_banks_captured": banks_captured,
            "distinct_failed_bank_capture_rate": float(banks_captured / total_failed_banks),
            "positive_bank_quarters_total": total_positive_quarters,
            "positive_bank_quarters_captured": positives_captured,
            "positive_bank_quarter_capture_rate": float(positives_captured / total_positive_quarters),
            "primary_capacity_result": "distinct_failed_bank_capture_rate",
            "secondary_capacity_result": "positive_bank_quarter_capture_rate",
        })
    return pd.DataFrame(summary_rows), pd.DataFrame(quarterly_rows), membership_masks


all_score_frames = {}
for model_id, predictions in test_prediction_frames.items():
    risk_frame = predictions[["panel_row_id", *PANEL_KEYS, "bootstrap_identity", TARGET]].copy()
    risk_frame["risk_score"] = predictions["predicted_probability"].astype("float64")
    all_score_frames[model_id] = risk_frame

# Low-equity baseline uses the lagged bank predictor and training-only preprocessing.
BASELINE_ID = "low_equity_to_assets_lag1_baseline"
BASELINE_FEATURE = "equity_to_assets_lag1"
baseline_training = train_data[BASELINE_FEATURE].astype("float64")
baseline_test = test_data[BASELINE_FEATURE].astype("float64")
baseline_median = float(baseline_training.median(skipna=True))
if not np.isfinite(baseline_median):
    raise AssertionError("The lagged equity baseline has no finite training median.")
baseline_training_imputed = baseline_training.fillna(baseline_median)
baseline_test_imputed = baseline_test.fillna(baseline_median)
baseline_lower = float(baseline_training_imputed.quantile(WINSOR_LOWER_QUANTILE, interpolation="linear"))
baseline_upper = float(baseline_training_imputed.quantile(WINSOR_UPPER_QUANTILE, interpolation="linear"))
baseline_test_winsorised = baseline_test_imputed.clip(lower=baseline_lower, upper=baseline_upper)
baseline_frame = test_data[["panel_row_id", *PANEL_KEYS, "bootstrap_identity", TARGET]].copy()
baseline_frame["risk_score"] = -baseline_test_winsorised.to_numpy(dtype="float64")
all_score_frames[BASELINE_ID] = baseline_frame
strict_json_dump({
    "baseline_id": BASELINE_ID, "feature": BASELINE_FEATURE,
    "ranking_direction": "lower lagged equity-to-assets means higher risk",
    "imputation_median_fitted_on_final_training": baseline_median,
    "winsor_lower_bound_fitted_on_final_training": baseline_lower,
    "winsor_upper_bound_fitted_on_final_training": baseline_upper,
    "test_outcome_used_to_define_baseline": False,
}, PREPROCESSING_DIR / "low_equity_to_assets_lag1_baseline.json")

capacity_parts, quarterly_capacity_parts = [], []
ranking_membership = {}
for source_id, frame in all_score_frames.items():
    source_type = "baseline" if source_id == BASELINE_ID else "model"
    summary, quarterly, masks = build_quarterly_rankings(frame, source_id, source_type)
    capacity_parts.append(summary)
    quarterly_capacity_parts.append(quarterly)
    for capacity, mask in masks.items():
        ranking_membership[(source_id, capacity)] = mask
test_capacity_summary = pd.concat(capacity_parts, ignore_index=True)
test_quarterly_capacity_results = pd.concat(quarterly_capacity_parts, ignore_index=True)

baseline_point_metrics = pd.DataFrame([{
    "source_id": BASELINE_ID, "source_type": "baseline",
    "test_observations": int(len(test_data)),
    "positive_bank_quarters": int(y_test.sum()),
    "distinct_failed_banks": int(test_data.loc[test_data[TARGET].eq(1), "bootstrap_identity"].nunique()),
    "roc_auc": float(roc_auc_score(y_test, baseline_frame["risk_score"])),
    "pr_auc": float(average_precision_score(y_test, baseline_frame["risk_score"])),
    "prevalence": float(y_test.mean()),
}])
baseline_point_metrics["pr_auc_divided_by_prevalence"] = baseline_point_metrics["pr_auc"] / baseline_point_metrics["prevalence"]

threshold_quarter_rows, threshold_summary_rows = [], []
for record in locked_models:
    model_id = str(record["model_id"])
    threshold = float(record["pooled_validation_f1_threshold"])
    frame = test_prediction_frames[model_id]
    flagged_all = frame["predicted_probability"].ge(threshold)
    for report_date, quarter in frame.groupby("report_date", sort=True):
        flagged = quarter["predicted_probability"].ge(threshold)
        threshold_quarter_rows.append({
            "model_id": model_id, "report_date": report_date,
            "locked_validation_threshold": threshold, "active_banks": int(len(quarter)),
            "observations_flagged": int(flagged.sum()), "percentage_flagged": float(flagged.mean()),
            "positive_bank_quarters": int(quarter[TARGET].sum()),
            "positive_bank_quarters_captured": int(quarter.loc[flagged, TARGET].sum()),
        })
    flagged = frame.loc[flagged_all]
    positive_captured = int(flagged[TARGET].sum())
    failed_captured = int(flagged.loc[flagged[TARGET].eq(1), "bootstrap_identity"].nunique())
    threshold_summary_rows.append({
        "model_id": model_id, "locked_validation_threshold": threshold,
        "test_observations": int(len(frame)), "test_quarters": EXPECTED_TEST_QUARTERS,
        "observations_flagged_total": int(flagged_all.sum()),
        "observations_flagged_mean_per_quarter": float(flagged_all.sum() / EXPECTED_TEST_QUARTERS),
        "percentage_flagged": float(flagged_all.mean()),
        "positive_bank_quarters_captured": positive_captured,
        "positive_bank_quarter_capture_rate": float(positive_captured / EXPECTED_TEST_POSITIVES),
        "distinct_failed_banks_captured": failed_captured,
        "distinct_failed_bank_capture_rate": float(failed_captured / EXPECTED_TEST_FAILED_BANKS),
        "recall": float(recall_score(y_test, flagged_all, zero_division=0)),
        "precision": float(precision_score(y_test, flagged_all, zero_division=0)),
        "f1": float(f1_score(y_test, flagged_all, zero_division=0)),
        "threshold_source": "new pooled Version 2 validation threshold; unchanged",
        "result_priority": "secondary",
        "prevalence_sensitivity_note": "The operating threshold may be sensitive to lower post-2020 failure prevalence.",
    })
test_quarterly_locked_threshold_results = pd.DataFrame(threshold_quarter_rows)
locked_threshold_test_results = pd.DataFrame(threshold_summary_rows)

test_capacity_summary.to_csv(NB16_OUTPUT_DIR / "v2_final_test_capacity_summary.csv", index=False)
test_quarterly_capacity_results.to_csv(NB16_OUTPUT_DIR / "v2_final_test_quarterly_capacity_results.csv", index=False)
baseline_point_metrics.to_csv(NB16_OUTPUT_DIR / "v2_low_equity_lag1_baseline_point_metrics.csv", index=False)
locked_threshold_test_results.to_csv(NB16_OUTPUT_DIR / "v2_locked_threshold_test_results.csv", index=False)
test_quarterly_locked_threshold_results.to_csv(NB16_OUTPUT_DIR / "v2_quarterly_locked_threshold_test_results.csv", index=False)

cell_05_checks = {
    "Six models and one lagged low-equity baseline use identical test rows": set(test_capacity_summary["source_id"]) == set(all_score_frames) and len(test_capacity_summary) == 7 * 4,
    "Every source has all four quarterly capacity groups": test_capacity_summary.groupby("source_id")["top_risk_percentage"].nunique().eq(4).all(),
    "Quarterly capacity output contains 616 rows": len(test_quarterly_capacity_results) == 7 * 4 * EXPECTED_TEST_QUARTERS,
    "Every quarterly group uses max(1, ceiling(capacity times active rows))": (test_quarterly_capacity_results["observations_flagged"] == np.maximum(1, np.ceil(test_quarterly_capacity_results["active_banks"] * test_quarterly_capacity_results["top_risk_percentage"])).astype(int)).all(),
    "Total flagged counts reconcile to mean per-quarter workload": np.allclose(test_capacity_summary["observations_flagged_mean_per_quarter"], test_capacity_summary["observations_flagged_total"] / EXPECTED_TEST_QUARTERS),
    "Distinct-failed-bank capture is primary and positive-bank-quarter capture secondary": test_capacity_summary["primary_capacity_result"].eq("distinct_failed_bank_capture_rate").all() and test_capacity_summary["secondary_capacity_result"].eq("positive_bank_quarter_capture_rate").all(),
    "Capture is monotonic as capacity expands": test_capacity_summary.sort_values(["source_id", "top_risk_percentage"]).groupby("source_id")[["distinct_failed_bank_capture_rate", "positive_bank_quarter_capture_rate"]].diff().fillna(0).ge(-1e-12).all().all(),
    "The baseline uses equity_to_assets_lag1 and outcome-blind training-only bounds": BASELINE_FEATURE == "equity_to_assets_lag1" and np.allclose(baseline_frame["risk_score"], -baseline_test_winsorised),
    "All six unchanged thresholds produce 132 quarterly results": len(test_quarterly_locked_threshold_results) == 6 * EXPECTED_TEST_QUARTERS and len(locked_threshold_test_results) == 6,
}
notebook_16_cell_05_gate = pd.DataFrame({
    "check": cell_05_checks.keys(),
    "result": ["PASS" if value else "FAIL" for value in cell_05_checks.values()],
    "passed": cell_05_checks.values(),
})
notebook_16_cell_05_gate.to_csv(NB16_OUTPUT_DIR / "cell_05_capacity_threshold_and_baseline_gate.csv", index=False)

print("Notebook 16 Cell 5 validation gate:")
display(notebook_16_cell_05_gate)
print("\nQuarterly capacity results with totals and mean per-quarter workload:")
display(test_capacity_summary[[
    "source_id", "top_risk_percentage", "observations_flagged_total",
    "observations_flagged_mean_per_quarter", "precision_in_top_group",
    "lift_over_test_prevalence", "distinct_failed_banks_captured",
    "distinct_failed_bank_capture_rate", "positive_bank_quarters_captured",
    "positive_bank_quarter_capture_rate",
]])
print("\nSecondary unchanged-threshold results:")
display(locked_threshold_test_results)
failed = [name for name, passed in cell_05_checks.items() if not passed]
if failed:
    raise AssertionError("Cell 5 blocked: " + "; ".join(failed))
print("\nCell 5 completed successfully.")

In [ ]:
# Compute common Version 1-identical CERT-clustered bootstrap confidence intervals

if not notebook_16_cell_05_gate["passed"].all():
    raise AssertionError("Cell 6 blocked because Cell 5 did not pass.")

# The Version 1 procedure is preserved exactly: sample bank identities with replacement,
# carry every bank-quarter through multiplicity weights, use 1,000 common multinomial
# resamples and percentile intervals. Verified CERT transitions have already been unified.
bank_ids = np.sort(test_data["bootstrap_identity"].unique())
bank_code = pd.Categorical(test_data["bootstrap_identity"], categories=bank_ids).codes.astype("int32")
n_banks = len(bank_ids)
rng = np.random.default_rng(BOOTSTRAP_SEED)
bootstrap_counts = np.empty((BOOTSTRAP_REPLICATES, n_banks), dtype="int16")
for replicate in range(BOOTSTRAP_REPLICATES):
    bootstrap_counts[replicate] = rng.multinomial(
        n_banks, np.full(n_banks, 1 / n_banks)
    ).astype("int16")

bootstrap_contract = pd.DataFrame({
    "item": [
        "identity", "identity mapping", "panel merge key", "replicates", "seed",
        "confidence level", "interval", "resampling", "paired comparison", "capacity ranking",
    ],
    "locked_value": [
        "CERT-derived audited bootstrap identity",
        "verified charter-conversion/merger CERT components mapped to one canonical identity",
        "IDRSSD plus report_date", BOOTSTRAP_REPLICATES, BOOTSTRAP_SEED,
        BOOTSTRAP_CONFIDENCE_LEVEL, "percentile",
        "sample test bank identities with replacement; carry all their bank-quarters using multiplicity weights",
        "same bootstrap multiplicities for every model and the baseline",
        "observed within-quarter memberships fixed; uncertainty is over sampled bank identities",
    ],
})
bootstrap_contract.to_csv(NB16_OUTPUT_DIR / "v2_bootstrap_design_contract.csv", index=False)


def prepare_weighted_discrimination(y, score):
    order = np.argsort(-score, kind="mergesort")
    sorted_score = score[order]
    sorted_y = y[order]
    starts = np.r_[0, np.flatnonzero(sorted_score[1:] != sorted_score[:-1]) + 1]
    return order, sorted_y, starts


def weighted_auc_and_ap(prepared, row_weights):
    order, sorted_y, starts = prepared
    sorted_weights = row_weights[order].astype("float64")
    positive_group = np.add.reduceat(sorted_weights * sorted_y, starts)
    negative_group = np.add.reduceat(sorted_weights * (1 - sorted_y), starts)
    total_positive = positive_group.sum()
    total_negative = negative_group.sum()
    if total_positive <= 0 or total_negative <= 0:
        return np.nan, np.nan
    cumulative_positive = np.cumsum(positive_group)
    cumulative_negative = np.cumsum(negative_group)
    tpr = np.r_[0.0, cumulative_positive / total_positive]
    fpr = np.r_[0.0, cumulative_negative / total_negative]
    roc = float(np.trapezoid(tpr, fpr) if hasattr(np, "trapezoid") else np.trapz(tpr, fpr))
    cumulative_weight = cumulative_positive + cumulative_negative
    precision = np.divide(
        cumulative_positive, cumulative_weight,
        out=np.zeros_like(cumulative_positive, dtype="float64"),
        where=cumulative_weight > 0,
    )
    average_precision = float(np.sum((positive_group / total_positive) * precision))
    return roc, average_precision


alpha = (1 - BOOTSTRAP_CONFIDENCE_LEVEL) / 2
discrimination_bootstrap_rows, discrimination_ci_rows = [], []
for source_number, (source_id, frame) in enumerate(all_score_frames.items(), start=1):
    score = frame["risk_score"].to_numpy(dtype="float64")
    prepared = prepare_weighted_discrimination(y_test, score)
    point_roc = float(roc_auc_score(y_test, score))
    point_pr = float(average_precision_score(y_test, score))
    unit_roc, unit_pr = weighted_auc_and_ap(prepared, np.ones(len(y_test), dtype="float64"))
    if not np.allclose([point_roc, point_pr], [unit_roc, unit_pr], atol=1e-12):
        raise AssertionError(f"Weighted discrimination implementation disagrees with sklearn for {source_id}.")
    source_rows = []
    for replicate in range(BOOTSTRAP_REPLICATES):
        row_weights = bootstrap_counts[replicate, bank_code]
        roc_value, pr_value = weighted_auc_and_ap(prepared, row_weights)
        row = {"source_id": source_id, "replicate": replicate + 1, "roc_auc": roc_value, "pr_auc": pr_value}
        source_rows.append(row)
        discrimination_bootstrap_rows.append(row)
    replicate_frame = pd.DataFrame(source_rows)
    discrimination_ci_rows.append({
        "source_id": source_id,
        "source_type": "baseline" if source_id == BASELINE_ID else "model",
        "validation_selected_overall_leader": source_id == LEADER_MODEL_ID,
        "test_observations": int(len(y_test)), "positive_bank_quarters": int(y_test.sum()),
        "distinct_failed_banks": EXPECTED_TEST_FAILED_BANKS,
        "roc_auc": point_roc,
        "roc_auc_ci_lower": float(replicate_frame["roc_auc"].quantile(alpha)),
        "roc_auc_ci_upper": float(replicate_frame["roc_auc"].quantile(1 - alpha)),
        "pr_auc": point_pr,
        "pr_auc_ci_lower": float(replicate_frame["pr_auc"].quantile(alpha)),
        "pr_auc_ci_upper": float(replicate_frame["pr_auc"].quantile(1 - alpha)),
        "prevalence": float(y_test.mean()),
        "pr_auc_divided_by_prevalence": float(point_pr / y_test.mean()),
        "bootstrap_identity": "audited CERT component",
        "bootstrap_seed": BOOTSTRAP_SEED,
        "bootstrap_replicates": BOOTSTRAP_REPLICATES,
    })
    print(f"[{source_number}/7] Discrimination bootstrap complete: {source_id}")

discrimination_bootstrap = pd.DataFrame(discrimination_bootstrap_rows)
final_test_discrimination_with_ci = pd.DataFrame(discrimination_ci_rows)

total_observations_by_bank = np.bincount(bank_code, minlength=n_banks).astype("float64")
positive_observations_by_bank = np.bincount(bank_code, weights=y_test, minlength=n_banks).astype("float64")
failed_bank = positive_observations_by_bank > 0
capacity_bootstrap_rows, capacity_ci_rows = [], []
for source_id in all_score_frames:
    for capacity in TOP_RISK_PERCENTAGES:
        flag = ranking_membership[(source_id, capacity)]
        flagged_observations_by_bank = np.bincount(bank_code, weights=flag.astype(float), minlength=n_banks)
        flagged_positive_by_bank = np.bincount(bank_code, weights=(flag & y_test.astype(bool)).astype(float), minlength=n_banks)
        captured_failed_bank = flagged_positive_by_bank > 0
        source_rows = []
        for replicate in range(BOOTSTRAP_REPLICATES):
            multiplicity = bootstrap_counts[replicate].astype("float64")
            observations = float(multiplicity @ total_observations_by_bank)
            positives = float(multiplicity @ positive_observations_by_bank)
            failed_banks = float(multiplicity @ failed_bank.astype(float))
            flagged_observations = float(multiplicity @ flagged_observations_by_bank)
            flagged_positive = float(multiplicity @ flagged_positive_by_bank)
            captured_banks = float(multiplicity @ captured_failed_bank.astype(float))
            precision = flagged_positive / flagged_observations if flagged_observations else np.nan
            prevalence = positives / observations if observations else np.nan
            row = {
                "source_id": source_id, "top_risk_percentage": capacity,
                "replicate": replicate + 1,
                "percentage_flagged": flagged_observations / observations if observations else np.nan,
                "precision_in_top_group": precision,
                "lift_over_test_prevalence": precision / prevalence if prevalence else np.nan,
                "distinct_failed_bank_capture_rate": captured_banks / failed_banks if failed_banks else np.nan,
                "positive_bank_quarter_capture_rate": flagged_positive / positives if positives else np.nan,
            }
            source_rows.append(row)
            capacity_bootstrap_rows.append(row)
        replicate_frame = pd.DataFrame(source_rows)
        point = test_capacity_summary.loc[
            test_capacity_summary["source_id"].eq(source_id)
            & test_capacity_summary["top_risk_percentage"].eq(capacity)
        ].iloc[0].to_dict()
        for metric in [
            "percentage_flagged", "precision_in_top_group", "lift_over_test_prevalence",
            "distinct_failed_bank_capture_rate", "positive_bank_quarter_capture_rate",
        ]:
            point[f"{metric}_ci_lower"] = float(replicate_frame[metric].quantile(alpha))
            point[f"{metric}_ci_upper"] = float(replicate_frame[metric].quantile(1 - alpha))
        point["bootstrap_identity"] = "audited CERT component"
        point["bootstrap_seed"] = BOOTSTRAP_SEED
        point["bootstrap_replicates"] = BOOTSTRAP_REPLICATES
        capacity_ci_rows.append(point)

capacity_bootstrap = pd.DataFrame(capacity_bootstrap_rows)
test_capacity_with_ci = pd.DataFrame(capacity_ci_rows)
discrimination_bootstrap.to_parquet(
    NB16_OUTPUT_DIR / "v2_discrimination_cluster_bootstrap_replicates.parquet",
    index=False, compression="snappy",
)
capacity_bootstrap.to_parquet(
    NB16_OUTPUT_DIR / "v2_capacity_cluster_bootstrap_replicates.parquet",
    index=False, compression="snappy",
)
final_test_discrimination_with_ci.to_csv(
    NB16_OUTPUT_DIR / "v2_final_test_discrimination_with_clustered_ci.csv", index=False
)
test_capacity_with_ci.to_csv(
    NB16_OUTPUT_DIR / "v2_final_test_capacity_with_clustered_ci.csv", index=False
)

cell_06_checks = {
    "The common bootstrap contains exactly 1,000 identity-cluster resamples": bootstrap_counts.shape == (BOOTSTRAP_REPLICATES, n_banks) and (bootstrap_counts.sum(axis=1) == n_banks).all(),
    "The Version 1 seed and multinomial resampling procedure are retained": BOOTSTRAP_SEED == 20260816 and BOOTSTRAP_REPLICATES == 1000,
    "Six models and the baseline each have 1,000 common discrimination replicates": len(discrimination_bootstrap) == 7 * BOOTSTRAP_REPLICATES and discrimination_bootstrap.groupby("source_id")["replicate"].nunique().eq(BOOTSTRAP_REPLICATES).all(),
    "All discrimination intervals are finite and ordered": len(final_test_discrimination_with_ci) == 7 and final_test_discrimination_with_ci[["roc_auc", "roc_auc_ci_lower", "roc_auc_ci_upper", "pr_auc", "pr_auc_ci_lower", "pr_auc_ci_upper"]].notna().all().all() and final_test_discrimination_with_ci["roc_auc_ci_lower"].le(final_test_discrimination_with_ci["roc_auc_ci_upper"]).all() and final_test_discrimination_with_ci["pr_auc_ci_lower"].le(final_test_discrimination_with_ci["pr_auc_ci_upper"]).all(),
    "All 28 capacity rows have 1,000 common clustered replicates": len(capacity_bootstrap) == 28 * BOOTSTRAP_REPLICATES and len(test_capacity_with_ci) == 28,
    "Models and baseline use the same audited identity, seed and replicate count": test_capacity_with_ci["bootstrap_identity"].eq("audited CERT component").all() and test_capacity_with_ci["bootstrap_seed"].eq(BOOTSTRAP_SEED).all() and test_capacity_with_ci["bootstrap_replicates"].eq(BOOTSTRAP_REPLICATES).all(),
}
notebook_16_cell_06_gate = pd.DataFrame({
    "check": cell_06_checks.keys(),
    "result": ["PASS" if value else "FAIL" for value in cell_06_checks.values()],
    "passed": cell_06_checks.values(),
})
notebook_16_cell_06_gate.to_csv(NB16_OUTPUT_DIR / "cell_06_bootstrap_gate.csv", index=False)

print("Notebook 16 Cell 6 validation gate:")
display(notebook_16_cell_06_gate)
print("\nFinal-test discrimination with 95% CERT-clustered bootstrap intervals:")
display(final_test_discrimination_with_ci)
print("\nCapacity results with 95% CERT-clustered bootstrap intervals:")
display(test_capacity_with_ci[[
    "source_id", "top_risk_percentage", "precision_in_top_group",
    "precision_in_top_group_ci_lower", "precision_in_top_group_ci_upper",
    "lift_over_test_prevalence", "distinct_failed_bank_capture_rate",
    "distinct_failed_bank_capture_rate_ci_lower", "distinct_failed_bank_capture_rate_ci_upper",
    "positive_bank_quarter_capture_rate", "positive_bank_quarter_capture_rate_ci_lower",
    "positive_bank_quarter_capture_rate_ci_upper",
]])
failed = [name for name, passed in cell_06_checks.items() if not passed]
if failed:
    raise AssertionError("Cell 6 blocked: " + "; ".join(failed))
print("\nCell 6 completed successfully.")

In [ ]:
# Compare pooled validation and test score distributions; report secondary calibration diagnostics

if not notebook_16_cell_06_gate["passed"].all():
    raise AssertionError("Cell 7 blocked because Cell 6 did not pass.")

validation_prediction_frames = {}
for model_id, rows in validation_source_manifest.groupby("model_id", sort=False):
    parts = []
    for row in rows.itertuples(index=False):
        source_path = Path(row.file)
        if not source_path.is_file():
            raise FileNotFoundError(f"Selected validation prediction file not found: {source_path}")
        input_hashes_before.setdefault(str(source_path), sha256_file(source_path))
        if hasattr(row, "expected_file_sha256") and sha256_file(source_path) != str(row.expected_file_sha256):
            raise AssertionError(f"Validation prediction hash changed: {source_path}")
        part = pd.read_parquet(
            source_path, columns=["report_date", "cert", "idrssd", TARGET, "predicted_probability"]
        )
        part["report_date"] = pd.to_datetime(part["report_date"], errors="raise").dt.normalize()
        parts.append(part)
    validation_prediction_frames[model_id] = pd.concat(parts, ignore_index=True).sort_values(
        PANEL_KEYS, kind="mergesort"
    ).reset_index(drop=True)

# Reconcile pooled counts to Notebook 15's source manifest instead of a notebook-local literal.
validation_expected_counts = (
    validation_source_manifest.groupby("model_id", as_index=False)
    .agg(
        expected_observations=("expected_observations", "sum"),
        expected_positive_bank_quarters=("expected_positive_bank_quarters", "sum"),
    )
)
validation_actual_counts = pd.DataFrame([
    {
        "model_id": model_id,
        "actual_observations": int(len(frame)),
        "actual_positive_bank_quarters": int(frame[TARGET].sum()),
    }
    for model_id, frame in validation_prediction_frames.items()
])
validation_count_reconciliation = validation_expected_counts.merge(
    validation_actual_counts, on="model_id", how="outer", validate="one_to_one"
)
validation_count_reconciliation["observations_match"] = (
    validation_count_reconciliation["actual_observations"]
    .eq(validation_count_reconciliation["expected_observations"])
)
validation_count_reconciliation["positive_bank_quarters_match"] = (
    validation_count_reconciliation["actual_positive_bank_quarters"]
    .eq(validation_count_reconciliation["expected_positive_bank_quarters"])
)


def distribution_record(model_id, sample_name, outcome_group, frame, threshold):
    if outcome_group == "positive":
        subset = frame.loc[frame[TARGET].eq(1)]
    elif outcome_group == "negative":
        subset = frame.loc[frame[TARGET].eq(0)]
    else:
        subset = frame
    score = subset["predicted_probability"].astype("float64")
    quantiles = score.quantile([0, 0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 1])
    return {
        "model_id": model_id, "sample": sample_name, "outcome_group": outcome_group,
        "observations": int(len(subset)), "mean": float(score.mean()), "std": float(score.std()),
        "minimum": float(quantiles.loc[0]), "p01": float(quantiles.loc[0.01]),
        "p05": float(quantiles.loc[0.05]), "p25": float(quantiles.loc[0.25]),
        "median": float(quantiles.loc[0.50]), "p75": float(quantiles.loc[0.75]),
        "p95": float(quantiles.loc[0.95]), "p99": float(quantiles.loc[0.99]),
        "maximum": float(quantiles.loc[1]), "locked_threshold": threshold,
        "number_above_threshold": int(score.ge(threshold).sum()),
        "percentage_above_threshold": float(score.ge(threshold).mean()),
    }


distribution_rows, distribution_comparison_rows = [], []
for record in locked_models:
    model_id = str(record["model_id"])
    threshold = float(record["pooled_validation_f1_threshold"])
    validation_frame = validation_prediction_frames[model_id]
    test_frame = test_prediction_frames[model_id]
    for outcome_group in ["all", "negative", "positive"]:
        distribution_rows.append(distribution_record(
            model_id, "pooled_out_of_time_validation", outcome_group, validation_frame, threshold
        ))
        distribution_rows.append(distribution_record(
            model_id, "final_test", outcome_group, test_frame, threshold
        ))
        if outcome_group == "all":
            validation_subset, test_subset = validation_frame, test_frame
        else:
            outcome = 1 if outcome_group == "positive" else 0
            validation_subset = validation_frame.loc[validation_frame[TARGET].eq(outcome)]
            test_subset = test_frame.loc[test_frame[TARGET].eq(outcome)]
        ks = ks_2samp(
            validation_subset["predicted_probability"], test_subset["predicted_probability"],
            alternative="two-sided", method="auto",
        )
        distribution_comparison_rows.append({
            "model_id": model_id, "outcome_group": outcome_group,
            "validation_observations": int(len(validation_subset)),
            "test_observations": int(len(test_subset)),
            "validation_mean_score": float(validation_subset["predicted_probability"].mean()),
            "test_mean_score": float(test_subset["predicted_probability"].mean()),
            "mean_score_change_test_minus_validation": float(test_subset["predicted_probability"].mean() - validation_subset["predicted_probability"].mean()),
            "kolmogorov_smirnov_statistic": float(ks.statistic),
            "kolmogorov_smirnov_p_value": float(ks.pvalue),
            "interpretation": "score-distribution diagnostic only; cannot alter the Version 2 lock",
        })
score_distribution_summary = pd.DataFrame(distribution_rows)
validation_test_score_comparison = pd.DataFrame(distribution_comparison_rows)


def calibration_diagnostics(model_id, frame, bins=10):
    outcomes = frame[TARGET].to_numpy(dtype="int8")
    probability = frame["predicted_probability"].to_numpy(dtype="float64")
    clipped = np.clip(probability, 1e-8, 1 - 1e-8)
    logits = np.log(clipped / (1 - clipped)).reshape(-1, 1)
    calibration_model = LogisticRegression(C=1e12, solver="lbfgs", max_iter=1000)
    calibration_model.fit(logits, outcomes)
    rank_bins = pd.qcut(
        pd.Series(probability).rank(method="first"), q=bins, labels=False, duplicates="drop"
    )
    curve = pd.DataFrame({
        "bin": rank_bins, "predicted_probability": probability, TARGET: outcomes,
    }).groupby("bin", as_index=False).agg(
        observations=(TARGET, "size"),
        mean_predicted_probability=("predicted_probability", "mean"),
        observed_failure_rate=(TARGET, "mean"), positives=(TARGET, "sum"),
    )
    curve["absolute_calibration_gap"] = (
        curve["mean_predicted_probability"] - curve["observed_failure_rate"]
    ).abs()
    ece = float((curve["observations"] / len(frame) * curve["absolute_calibration_gap"]).sum())
    summary = {
        "model_id": model_id, "observations": int(len(frame)),
        "positive_bank_quarters": int(outcomes.sum()),
        "mean_predicted_probability": float(probability.mean()),
        "observed_prevalence": float(outcomes.mean()),
        "mean_score_divided_by_prevalence": float(probability.mean() / outcomes.mean()),
        "brier_score": float(brier_score_loss(outcomes, probability)),
        "log_loss": float(log_loss(outcomes, clipped, labels=[0, 1])),
        "calibration_intercept": float(calibration_model.intercept_[0]),
        "calibration_slope": float(calibration_model.coef_[0, 0]),
        "expected_calibration_error_10_equal_frequency_bins": ece,
        "diagnostic_priority": "secondary; no recalibration and no effect on the model lock",
    }
    curve.insert(0, "model_id", model_id)
    return summary, curve


calibration_rows, calibration_curve_parts = [], []
for model_id, frame in test_prediction_frames.items():
    summary, curve = calibration_diagnostics(model_id, frame)
    calibration_rows.append(summary)
    calibration_curve_parts.append(curve)
calibration_diagnostic_summary = pd.DataFrame(calibration_rows)
calibration_curve_table = pd.concat(calibration_curve_parts, ignore_index=True)

score_distribution_summary.to_csv(NB16_OUTPUT_DIR / "v2_validation_and_test_score_distributions.csv", index=False)
validation_test_score_comparison.to_csv(NB16_OUTPUT_DIR / "v2_validation_test_score_distribution_comparison.csv", index=False)
calibration_diagnostic_summary.to_csv(NB16_OUTPUT_DIR / "v2_calibration_diagnostic_summary.csv", index=False)
calibration_curve_table.to_csv(NB16_OUTPUT_DIR / "v2_calibration_curve_deciles.csv", index=False)
validation_count_reconciliation.to_csv(
    NB16_OUTPUT_DIR / "cell_07_validation_count_reconciliation.csv", index=False
)

cell_07_checks = {
    "All six pooled validation panels reconcile exactly to the Notebook 15 source manifest": len(validation_prediction_frames) == 6 and len(validation_count_reconciliation) == 6 and validation_count_reconciliation[["observations_match", "positive_bank_quarters_match"]].all().all(),
    "Validation and test distributions cover all, negative and positive observations": len(score_distribution_summary) == 6 * 2 * 3 and score_distribution_summary.groupby(["model_id", "sample"])["outcome_group"].nunique().eq(3).all(),
    "Exactly 18 validation-test comparisons are diagnostic only": len(validation_test_score_comparison) == 18 and validation_test_score_comparison["interpretation"].str.contains("cannot alter", case=False).all(),
    "Calibration is reported only for the six probability models": len(calibration_diagnostic_summary) == 6 and calibration_curve_table["model_id"].nunique() == 6,
    "Every calibration result is finite and explicitly secondary": np.isfinite(calibration_diagnostic_summary[["brier_score", "log_loss", "calibration_intercept", "calibration_slope", "expected_calibration_error_10_equal_frequency_bins"]]).all().all() and calibration_diagnostic_summary["diagnostic_priority"].str.contains("secondary", case=False).all(),
    "No diagnostic changes the locked validation leader": LEADER_MODEL_ID == "random_forest__bank_plus_macro",
}
notebook_16_cell_07_gate = pd.DataFrame({
    "check": cell_07_checks.keys(),
    "result": ["PASS" if value else "FAIL" for value in cell_07_checks.values()],
    "passed": cell_07_checks.values(),
})
notebook_16_cell_07_gate.to_csv(NB16_OUTPUT_DIR / "cell_07_distribution_and_calibration_gate.csv", index=False)

print("Notebook 16 Cell 7 validation gate:")
display(notebook_16_cell_07_gate)
print("\nPooled validation count reconciliation to Notebook 15:")
display(validation_count_reconciliation)
print("\nValidation versus test score-distribution comparison:")
display(validation_test_score_comparison)
print("\nSecondary calibration diagnostics for class-weighted model scores:")
display(calibration_diagnostic_summary)
failed = [name for name, passed in cell_07_checks.items() if not passed]
if failed:
    raise AssertionError("Cell 7 blocked: " + "; ".join(failed))
print("\nCell 7 completed successfully.")

In [ ]:
# Produce dissertation-facing tables, figures and the immutable Notebook 17 handoff

if not notebook_16_cell_07_gate["passed"].all():
    raise AssertionError("Cell 8 blocked because Cell 7 did not pass.")

source_order = [record["model_id"] for record in sorted(locked_models, key=lambda item: item["model_id"])] + [BASELINE_ID]
discrimination_table = final_test_discrimination_with_ci.copy()
discrimination_table["source_id"] = pd.Categorical(
    discrimination_table["source_id"], categories=source_order, ordered=True
)
discrimination_table.sort_values("source_id", inplace=True)
discrimination_table["roc_auc_95ci"] = discrimination_table.apply(
    lambda row: f'{row.roc_auc:.6f} [{row.roc_auc_ci_lower:.6f}, {row.roc_auc_ci_upper:.6f}]', axis=1
)
discrimination_table["pr_auc_95ci"] = discrimination_table.apply(
    lambda row: f'{row.pr_auc:.6f} [{row.pr_auc_ci_lower:.6f}, {row.pr_auc_ci_upper:.6f}]', axis=1
)
dissertation_discrimination = discrimination_table[[
    "source_id", "validation_selected_overall_leader", "test_observations",
    "positive_bank_quarters", "distinct_failed_banks", "roc_auc_95ci",
    "pr_auc_95ci", "pr_auc_divided_by_prevalence",
]].copy()

dissertation_capacity = test_capacity_with_ci[[
    "source_id", "top_risk_percentage", "observations_flagged_total",
    "observations_flagged_mean_per_quarter", "precision_in_top_group",
    "precision_in_top_group_ci_lower", "precision_in_top_group_ci_upper",
    "lift_over_test_prevalence", "lift_over_test_prevalence_ci_lower",
    "lift_over_test_prevalence_ci_upper", "distinct_failed_banks_total",
    "distinct_failed_banks_captured", "distinct_failed_bank_capture_rate",
    "distinct_failed_bank_capture_rate_ci_lower", "distinct_failed_bank_capture_rate_ci_upper",
    "positive_bank_quarters_total", "positive_bank_quarters_captured",
    "positive_bank_quarter_capture_rate", "positive_bank_quarter_capture_rate_ci_lower",
    "positive_bank_quarter_capture_rate_ci_upper",
]].copy()
dissertation_threshold = locked_threshold_test_results.copy()
dissertation_discrimination.to_csv(NB16_OUTPUT_DIR / "table_16_1_final_test_discrimination.csv", index=False)
dissertation_capacity.to_csv(NB16_OUTPUT_DIR / "table_16_2_quarterly_capacity.csv", index=False)
dissertation_threshold.to_csv(NB16_OUTPUT_DIR / "table_16_3_secondary_locked_threshold.csv", index=False)
predictor_horizon_summary.to_csv(NB16_OUTPUT_DIR / "table_16_4_realised_predictor_horizon.csv", index=False)

reporting_statements = pd.DataFrame({
    "topic": [
        "test calendar", "Version 2 precommitment", "selection", "threshold",
        "calibration", "horizon", "Version comparison",
    ],
    "required_statement": [
        "The 2020Q1–2025Q2 calendar was examined under archived unlagged Version 1 and is not described as never viewed.",
        "The lagged design and complete Version 2 selection procedure were fixed before Version 2 evaluation.",
        f"{LEADER_MODEL_ID} remains the validation-selected leader; no test-based reselection was performed.",
        "The pooled Version 2 validation F1 threshold was applied unchanged as a secondary result and may be prevalence-sensitive.",
        "Calibration is secondary because class-weighted model scores are not necessarily calibrated probabilities.",
        "The realised source-report-to-failure horizon is longer than Version 1 although the outcome remains (t,t+365].",
        "Version 1–Version 2 differences reflect timing, effective-horizon and eligible-sample changes and are not a pure causal timing effect.",
    ],
})
reporting_statements.to_csv(NB16_OUTPUT_DIR / "notebook_16_required_reporting_statements.csv", index=False)

# Figure 1: discrimination point estimates and clustered confidence intervals.
plot_frame = discrimination_table.reset_index(drop=True)
positions = np.arange(len(plot_frame))
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
for axis, metric, label in [
    (axes[0], "pr_auc", "PR-AUC"), (axes[1], "roc_auc", "ROC-AUC"),
]:
    lower = plot_frame[metric] - plot_frame[f"{metric}_ci_lower"]
    upper = plot_frame[f"{metric}_ci_upper"] - plot_frame[metric]
    colours = ["#C44E52" if value == LEADER_MODEL_ID else "#4C72B0" for value in plot_frame["source_id"].astype(str)]
    axis.errorbar(positions, plot_frame[metric], yerr=np.vstack([lower, upper]), fmt="none", ecolor="#333333", capsize=3)
    axis.scatter(positions, plot_frame[metric], c=colours, s=55, zorder=3)
    axis.set_xticks(positions)
    axis.set_xticklabels(plot_frame["source_id"].astype(str), rotation=55, ha="right")
    axis.set_ylabel(label)
    axis.set_title(f"Final-test {label} with 95% clustered CI")
    axis.grid(axis="y", alpha=0.25)
fig.savefig(FIGURE_DIR / "figure_16_1_discrimination_with_clustered_ci.png", dpi=220, bbox_inches="tight")
plt.show()

# Figure 2: validation-selected leader versus lagged low-equity baseline at fixed capacities.
capacity_plot = test_capacity_with_ci.loc[
    test_capacity_with_ci["source_id"].isin([LEADER_MODEL_ID, BASELINE_ID])
].copy()
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
for source_id, group in capacity_plot.groupby("source_id", sort=False):
    group = group.sort_values("top_risk_percentage")
    axes[0].plot(group["top_risk_percentage"] * 100, group["distinct_failed_bank_capture_rate"], marker="o", label=source_id)
    axes[1].plot(group["top_risk_percentage"] * 100, group["positive_bank_quarter_capture_rate"], marker="o", label=source_id)
axes[0].set_title("Primary: distinct failed-bank capture")
axes[1].set_title("Secondary: positive bank-quarter capture")
for axis in axes:
    axis.set_xlabel("Quarterly capacity (%)")
    axis.set_ylabel("Capture rate")
    axis.set_ylim(0, 1.05)
    axis.grid(alpha=0.25)
    axis.legend(fontsize=8)
fig.savefig(FIGURE_DIR / "figure_16_2_leader_vs_lagged_equity_capacity.png", dpi=220, bbox_inches="tight")
plt.show()

# Figure 3: secondary calibration diagnostic for the validation-selected leader.
leader_curve = calibration_curve_table.loc[calibration_curve_table["model_id"].eq(LEADER_MODEL_ID)]
fig, ax = plt.subplots(figsize=(6, 5), constrained_layout=True)
ax.plot(leader_curve["mean_predicted_probability"], leader_curve["observed_failure_rate"], marker="o", label=LEADER_MODEL_ID)
limit = max(float(leader_curve["mean_predicted_probability"].max()), float(leader_curve["observed_failure_rate"].max()))
ax.plot([0, limit], [0, limit], linestyle="--", color="#555555", label="45-degree reference")
ax.set_xlabel("Mean model score")
ax.set_ylabel("Observed failure rate")
ax.set_title("Secondary calibration diagnostic")
ax.grid(alpha=0.25)
ax.legend(fontsize=8)
fig.savefig(FIGURE_DIR / "figure_16_3_leader_calibration_diagnostic.png", dpi=220, bbox_inches="tight")
plt.show()

# Figure 4: pooled validation versus final-test score distributions for the leader.
leader_validation = validation_prediction_frames[LEADER_MODEL_ID]["predicted_probability"].astype("float64")
leader_test = test_prediction_frames[LEADER_MODEL_ID]["predicted_probability"].astype("float64")
fig, ax = plt.subplots(figsize=(7, 4.5), constrained_layout=True)
ax.hist(leader_validation, bins=50, density=True, alpha=0.45, label="Pooled out-of-time validation")
ax.hist(leader_test, bins=50, density=True, alpha=0.45, label="Final test")
ax.set_xlabel("Predicted score")
ax.set_ylabel("Density")
ax.set_title("Validation-selected leader: score-distribution shift")
ax.legend()
fig.savefig(FIGURE_DIR / "figure_16_4_leader_validation_test_score_distribution.png", dpi=220, bbox_inches="tight")
plt.show()

notebook_17_handoff = {
    "schema_version": "v2_notebook_16_handoff_1.0",
    "analysis_version": ANALYSIS_VERSION,
    "run_signature": RUN_SIGNATURE,
    "validation_selected_leader": LEADER_MODEL_ID,
    "test_based_reselection_performed": False,
    "test_calendar": FINAL_TEST_DESIGN,
    "test_observations": int(len(test_data)),
    "positive_bank_quarters": EXPECTED_TEST_POSITIVES,
    "distinct_failed_banks": EXPECTED_TEST_FAILED_BANKS,
    "bootstrap": {
        "identity": "audited CERT component", "seed": BOOTSTRAP_SEED,
        "replicates": BOOTSTRAP_REPLICATES, "confidence_level": BOOTSTRAP_CONFIDENCE_LEVEL,
        "procedure": "Version 1-identical multinomial cluster multiplicity bootstrap",
    },
    "capacity_fractions": list(TOP_RISK_PERCENTAGES),
    "primary_capacity_result": "distinct failed-bank capture",
    "secondary_capacity_result": "positive bank-quarter capture",
    "baseline": BASELINE_ID,
    "horizon_statement": "Longer than Version 1 because bank predictors are measured at t-1 while the outcome remains (t,t+365].",
    "principal_outputs": {
        "predictions_directory": str(PREDICTION_DIR),
        "discrimination": str(NB16_OUTPUT_DIR / "v2_final_test_discrimination_with_clustered_ci.csv"),
        "capacity": str(NB16_OUTPUT_DIR / "v2_final_test_capacity_with_clustered_ci.csv"),
        "threshold": str(NB16_OUTPUT_DIR / "v2_locked_threshold_test_results.csv"),
        "horizon": str(NB16_OUTPUT_DIR / "realised_predictor_date_to_failure_horizon.csv"),
    },
}
strict_json_dump(notebook_17_handoff, NB16_OUTPUT_DIR / "v2_notebook_16_to_17_handoff.json")

cell_08_checks = {
    "All four dissertation tables were saved": all((NB16_OUTPUT_DIR / name).is_file() for name in ["table_16_1_final_test_discrimination.csv", "table_16_2_quarterly_capacity.csv", "table_16_3_secondary_locked_threshold.csv", "table_16_4_realised_predictor_horizon.csv"]),
    "All four final-test figures were saved": all((FIGURE_DIR / name).is_file() for name in ["figure_16_1_discrimination_with_clustered_ci.png", "figure_16_2_leader_vs_lagged_equity_capacity.png", "figure_16_3_leader_calibration_diagnostic.png", "figure_16_4_leader_validation_test_score_distribution.png"]),
    "The reporting file discloses prior Version 1 calendar examination": reporting_statements["required_statement"].str.contains("examined under archived", case=False).any(),
    "The handoff preserves the validation leader without test reselection": notebook_17_handoff["validation_selected_leader"] == LEADER_MODEL_ID and notebook_17_handoff["test_based_reselection_performed"] is False,
    "The handoff records the longer realised predictor horizon": "Longer than Version 1" in notebook_17_handoff["horizon_statement"],
}
notebook_16_cell_08_gate = pd.DataFrame({
    "check": cell_08_checks.keys(),
    "result": ["PASS" if value else "FAIL" for value in cell_08_checks.values()],
    "passed": cell_08_checks.values(),
})
notebook_16_cell_08_gate.to_csv(NB16_OUTPUT_DIR / "cell_08_reporting_and_handoff_gate.csv", index=False)

print("Notebook 16 Cell 8 validation gate:")
display(notebook_16_cell_08_gate)
print("\nDissertation-facing discrimination table:")
display(dissertation_discrimination)
print("\nRequired reporting statements:")
with pd.option_context("display.max_colwidth", None):
    display(reporting_statements)
failed = [name for name, passed in cell_08_checks.items() if not passed]
if failed:
    raise AssertionError("Cell 8 blocked: " + "; ".join(failed))
print("\nCell 8 completed successfully.")

In [ ]:
# Prove input immutability, lock the completed run and save the final output manifest

if not notebook_16_cell_08_gate["passed"].all():
    raise AssertionError("Cell 9 blocked because Cell 8 did not pass.")

input_immutability_rows = []
for path_string, before_hash in input_hashes_before.items():
    path = Path(path_string)
    after_hash = sha256_file(path)
    input_immutability_rows.append({
        "input_file": path_string, "sha256_before": before_hash,
        "sha256_after": after_hash, "unchanged": before_hash == after_hash,
    })
input_immutability = pd.DataFrame(input_immutability_rows)
input_immutability.to_csv(NB16_OUTPUT_DIR / "cell_09_input_immutability_audit.csv", index=False)

earlier_gates = [
    notebook_16_cell_01_gate, notebook_16_cell_02_gate, notebook_16_cell_03_gate,
    notebook_16_cell_04_gate, notebook_16_cell_05_gate, notebook_16_cell_06_gate,
    notebook_16_cell_07_gate, notebook_16_cell_08_gate,
]
final_summary = pd.DataFrame({
    "measure": [
        "Analysis version", "Training period", "Purge period", "Test period",
        "Test observations", "Positive bank-quarters", "Distinct failed banks",
        "Validation-selected leader", "Test-based reselection performed",
        "Models evaluated", "Bootstrap identity", "Bootstrap seed", "Bootstrap replicates",
        "Capacity fractions", "Primary capacity result", "Secondary capacity result",
        "Threshold source", "Calibration priority", "Baseline",
        "Effective horizon", "Prior Version 1 calendar examination disclosed",
        "Input files modified", "Next notebook",
    ],
    "value": [
        ANALYSIS_VERSION, "2001Q2–2018Q4", "2019Q1–2019Q4", "2020Q1–2025Q2",
        int(len(test_data)), EXPECTED_TEST_POSITIVES, EXPECTED_TEST_FAILED_BANKS,
        LEADER_MODEL_ID, False, 6, "audited CERT component", BOOTSTRAP_SEED,
        BOOTSTRAP_REPLICATES, str(TOP_RISK_PERCENTAGES), "distinct failed-bank capture",
        "positive bank-quarter capture", "pooled Version 2 validation; unchanged",
        "secondary", BASELINE_ID,
        "realised source_report_date-to-failure days; longer than Version 1",
        True, int((~input_immutability["unchanged"]).sum()),
        "17_Robustness_Interpretation_and_Final_Outputs.ipynb",
    ],
})
final_summary.to_csv(NB16_OUTPUT_DIR / "cell_09_final_summary.csv", index=False)

final_checks = {
    "All eight earlier Notebook 16 gates passed": all(gate["passed"].all() for gate in earlier_gates),
    "Every loaded input remained byte-for-byte unchanged": input_immutability["unchanged"].all(),
    "Exactly six final-test prediction files exist": len(list(PREDICTION_DIR.glob("*__v2_final_test_predictions.parquet"))) == 6,
    "All six models and the baseline have clustered discrimination intervals": len(final_test_discrimination_with_ci) == 7,
    "All 28 quarterly capacity results have clustered intervals": len(test_capacity_with_ci) == 28,
    "The pooled Version 2 thresholds remained unchanged and secondary": len(locked_threshold_test_results) == 6 and locked_threshold_test_results["result_priority"].eq("secondary").all(),
    "Calibration remained a secondary diagnostic": calibration_diagnostic_summary["diagnostic_priority"].str.contains("secondary", case=False).all(),
    "The realised predictor horizon is explicitly longer than Version 1": predictor_horizon_summary["interpretation"].str.contains("Longer than Version 1", case=False).all(),
    "The same test calendar's earlier Version 1 examination is disclosed": reporting_statements["required_statement"].str.contains("examined under archived", case=False).any(),
    "No test-based reselection or relocking was performed": notebook_17_handoff["test_based_reselection_performed"] is False and LEADER_MODEL_ID == model_lock_manifest["selection"]["leader_model_id"],
}
notebook_16_cell_09_gate = pd.DataFrame({
    "check": final_checks.keys(),
    "result": ["PASS" if value else "FAIL" for value in final_checks.values()],
    "passed": final_checks.values(),
})
notebook_16_cell_09_gate.to_csv(NB16_OUTPUT_DIR / "cell_09_final_validation_gate.csv", index=False)

principal_purposes = {
    "cell_01_final_evaluation_contract.csv": "Locked one-time Version 2 evaluation contract",
    "cell_02_final_role_summary.csv": "Final training, purge and test sample summary",
    "cell_03_final_samples.csv": "Loaded training/test samples and purge non-load audit",
    "v2_cert_to_bootstrap_identity_audit.csv": "Verified CERT-transition bootstrap identity mapping",
    "realised_predictor_date_to_failure_horizon.csv": "Realised lagged predictor-date horizon",
    "cell_04_final_fit_register.csv": "Six locked final fits and training-only weight audit",
    "v2_final_test_point_metrics.csv": "Six-model final-test point metrics and unchanged thresholds",
    "v2_final_test_discrimination_with_clustered_ci.csv": "Model and baseline ROC-AUC/PR-AUC with clustered intervals",
    "v2_final_test_capacity_with_clustered_ci.csv": "Quarterly capacity results with clustered intervals",
    "v2_locked_threshold_test_results.csv": "Secondary unchanged-threshold results",
    "cell_07_validation_count_reconciliation.csv": "Pooled validation row and positive counts reconciled to Notebook 15",
    "v2_validation_test_score_distribution_comparison.csv": "Validation-test score shift diagnostics",
    "v2_calibration_diagnostic_summary.csv": "Secondary calibration diagnostics",
    "table_16_1_final_test_discrimination.csv": "Dissertation-facing discrimination table",
    "table_16_2_quarterly_capacity.csv": "Dissertation-facing capacity table",
    "table_16_3_secondary_locked_threshold.csv": "Dissertation-facing threshold table",
    "table_16_4_realised_predictor_horizon.csv": "Dissertation-facing horizon table",
    "notebook_16_required_reporting_statements.csv": "Mandatory cautious reporting language",
    "v2_notebook_16_to_17_handoff.json": "Immutable Notebook 17 handoff",
    "cell_09_input_immutability_audit.csv": "Before/after hashes of all loaded inputs",
    "cell_09_final_summary.csv": "Final Notebook 16 summary",
    "cell_09_final_validation_gate.csv": "Final fail-fast gate",
}
principal_paths = [NB16_OUTPUT_DIR / name for name in principal_purposes]
missing_principal = [path for path in principal_paths if not path.is_file()]
if missing_principal:
    raise AssertionError("Missing principal Notebook 16 outputs: " + "; ".join(map(str, missing_principal)))
output_manifest = pd.DataFrame([{
    "file": str(path.relative_to(NB16_OUTPUT_DIR)),
    "purpose": principal_purposes[path.name],
    "bytes": int(path.stat().st_size), "sha256": sha256_file(path),
} for path in principal_paths])
output_manifest.to_csv(NB16_OUTPUT_DIR / "cell_09_output_manifest.csv", index=False)

failed = [name for name, passed in final_checks.items() if not passed]
if failed:
    raise AssertionError("Cell 9 blocked: " + "; ".join(failed))

complete_marker = {
    "status": "COMPLETE", "analysis_version": ANALYSIS_VERSION,
    "run_signature": RUN_SIGNATURE, "completed_utc": datetime.now(timezone.utc).isoformat(),
    "model_lock_sha256": lock_sha256, "membership_key_sha256": membership_key_sha256,
    "test_observations": int(len(test_data)),
    "positive_bank_quarters": EXPECTED_TEST_POSITIVES,
    "distinct_failed_banks": EXPECTED_TEST_FAILED_BANKS,
    "validation_selected_leader": LEADER_MODEL_ID,
    "test_based_reselection_performed": False,
    "output_manifest_sha256": sha256_file(NB16_OUTPUT_DIR / "cell_09_output_manifest.csv"),
}
strict_json_dump(complete_marker, RUN_MARKER_PATH)

print("Notebook 16 Cell 9 final validation gate:")
display(notebook_16_cell_09_gate)
print("\nNotebook 16 final summary:")
with pd.option_context("display.max_colwidth", None):
    display(final_summary)
print("\nPrincipal output manifest:")
display(output_manifest)
print("\nNotebook 16 is PASS / FINAL VERSION 2 TEST LOCKED.")
print("Validation-selected leader retained:", LEADER_MODEL_ID)
print("Test-based reselection performed: False")
print("Notebook 17 handoff:", NB16_OUTPUT_DIR / "v2_notebook_16_to_17_handoff.json")